# DRP-AP: Directional-Rank-Profile Adaptive Partitioning — reproducible Colab

Implements the paper *Directional-Rank Deficiency: A Unified Failure Mode of PCA-Part and
Mahalanobis k-means++ with a Principled Fix*: the **DRP-AP** initializer, 10 baselines, the
12-dataset protocol (tabular, image, text, audio, sensor), synthetic generators with controlled
directional rank, ablations A1–A7, sensitivity grid, seed stability, and Friedman/Nemenyi +
Wilcoxon–Holm statistics.

**How to use:** *Runtime → Run all.* The notebook is self-contained (it writes the package files
itself). Start with `QUICK = True` (≈ 1 min); then set `QUICK = False` for the full protocol.
GPU is not needed.

In [ ]:
!pip -q install diptest pillow
!mkdir -p drp_ap tests tools results data_cache

## 1. Package source (written to the Colab disk)

In [ ]:
%%writefile drp_ap/__init__.py
"""DRP-AP: Directional-Rank-Profile Adaptive Partitioning for k-means initialisation."""
from .core import DRPAP, DRPAPSA, DRPAPSAR, directional_rank, conditioning_index  # noqa: F401

from .fuzzy import FuzzyDRPAPSA, fcm  # noqa: F401

__version__ = "1.0.0"

In [ ]:
%%writefile drp_ap/core.py
"""DRP-AP: Directional-Rank-Profile Adaptive Partitioning (k-means initializer).

Implements Algorithm 1 of the paper:
  * Mode A  (r_delta = 1)                       -> PCA-Part split along u_1
  * Mode B  (r_delta >= 2, well conditioned)    -> Mahalanobis 2-means in span(u_1..u_r)
  * Mode C  (r_delta >= 2, ill conditioned)     -> same, with shrinkage-regularised metric
Options expose every ablation of the protocol (A1-A5) and the DRP-AP+ variant.
"""
from __future__ import annotations

import numpy as np
from sklearn.covariance import ledoit_wolf_shrinkage
from sklearn.covariance import oas as _oas
from sklearn.metrics import pairwise_distances_argmin

# ----------------------------------------------------------------------------
# Spectral diagnostics
# ----------------------------------------------------------------------------


def centered_sse(Xc: np.ndarray) -> float:
    return float(np.einsum("ij,ij->", Xc, Xc))


def spectrum(Xc: np.ndarray):
    """Eigenvalues (biased covariance) and eigenvectors (rows of Vt) of a centred block.

    Uses the SVD of the data matrix, so it also works when m <= d (Gram regime).
    Returns lam (length min(m, d)) sorted decreasingly and Vt (min(m,d) x d).
    """
    m = Xc.shape[0]
    _, s, Vt = np.linalg.svd(Xc, full_matrices=False)
    return (s ** 2) / m, Vt


def directional_rank(lam: np.ndarray, delta: float) -> int:
    """r_delta(C): smallest r with sum_{j<=r} lam_j / sum_j lam_j >= 1 - delta."""
    tot = float(lam.sum())
    if tot <= 0.0:
        return 1
    c = np.cumsum(lam) / tot
    r = int(np.searchsorted(c, 1.0 - delta - 1e-12, side="left")) + 1
    return int(min(max(r, 1), len(lam)))


def conditioning_index(lam: np.ndarray, m: int, d: int, tiny: float = 1e-14) -> float:
    """kappa(C) = lam_1 / lam_min^+ ; infinity when the covariance is rank deficient (m <= d)."""
    if m <= d or lam[0] <= 0.0:
        return float("inf")
    lmin = lam[-1]
    if lmin <= tiny * lam[0]:
        return float("inf")
    return float(lam[0] / lmin)


def shrinkage_alpha(Xc: np.ndarray, rule: str) -> float:
    m = Xc.shape[0]
    if m < 3:
        return 1.0
    if rule == "lw":
        return float(ledoit_wolf_shrinkage(Xc, assume_centered=True))
    if rule == "oas":
        return float(_oas(Xc, assume_centered=True)[1])
    return 0.0


# ----------------------------------------------------------------------------
# Unimodality test (for DRP-AP+)
# ----------------------------------------------------------------------------
_DIP = None


def unimodality_pvalue(x: np.ndarray) -> float:
    """Hartigan dip-test p-value (package `diptest`); falls back to a 1-vs-2 Gaussian BIC test."""
    global _DIP
    if _DIP is None:
        try:
            import diptest  # type: ignore

            _DIP = diptest
        except Exception:  # pragma: no cover
            _DIP = False
    x = np.asarray(x, dtype=float).ravel()
    if len(x) < 8 or np.ptp(x) <= 0:
        return 1.0
    if _DIP:
        try:
            return float(_DIP.diptest(x)[1])
        except Exception:  # pragma: no cover
            pass
    from sklearn.mixture import GaussianMixture

    xs = x.reshape(-1, 1)
    b1 = GaussianMixture(1, random_state=0).fit(xs).bic(xs)
    b2 = GaussianMixture(2, random_state=0, n_init=2).fit(xs).bic(xs)
    return 0.01 if b2 < b1 - 10.0 else 1.0


# ----------------------------------------------------------------------------
# Small helpers
# ----------------------------------------------------------------------------


def median_split(proj: np.ndarray, how: str = "median"):
    """Split indices (relative to proj) into two non-empty parts."""
    if how == "mean":
        mask = proj > proj.mean()
        if 0 < mask.sum() < len(proj):
            return np.where(~mask)[0], np.where(mask)[0]
    order = np.argsort(proj, kind="stable")
    h = max(len(proj) // 2, 1)
    return order[:h], order[h:]


def kpp_indices(Z: np.ndarray, k: int, rng: np.random.Generator):
    """D^2 sampling (k-means++) of k indices under the Euclidean metric of Z."""
    n = len(Z)
    idx = [int(rng.integers(n))]
    d2 = ((Z - Z[idx[0]]) ** 2).sum(1)
    for _ in range(1, k):
        s = float(d2.sum())
        i = int(rng.integers(n)) if s <= 0 else int(rng.choice(n, p=d2 / s))
        idx.append(i)
        d2 = np.minimum(d2, ((Z - Z[i]) ** 2).sum(1))
    return idx


def lloyd_in_space(Z: np.ndarray, centers: np.ndarray, T: int):
    """T Lloyd iterations. Returns labels (or None if a cluster becomes empty)."""
    k = len(centers)
    labels = pairwise_distances_argmin(Z, centers)
    for _ in range(T):
        new_centers = centers.copy()
        for j in range(k):
            sel = labels == j
            if not sel.any():
                return None
            new_centers[j] = Z[sel].mean(0)
        new_labels = pairwise_distances_argmin(Z, new_centers)
        centers = new_centers
        if np.array_equal(new_labels, labels):
            break
        labels = new_labels
    if len(np.unique(labels)) < k:
        return None
    return labels


# ----------------------------------------------------------------------------
# DRP-AP
# ----------------------------------------------------------------------------


class DRPAP:
    """Directional-Rank-Profile Adaptive Partitioning.

    Parameters
    ----------
    delta        : variance fraction defining r_delta (default 0.05). delta=1 => PCA-Part (A1).
    kappa0       : conditioning threshold switching Mode B -> Mode C (default 1e3).
    shrinkage    : 'lw' (Ledoit-Wolf), 'oas', 'eps' (fixed ridge eps_rel * tr/d), 'none'.
    eps_rel      : ridge level for shrinkage='eps'.
    local_iters  : T, Lloyd iterations inside Modes B/C (default 5).
    adaptive_k   : DRP-AP+ (number of sub-clusters from the dip test on projections).
    metric       : 'mahalanobis' (default) or 'euclid' in Modes B/C (ablation A5).
    mode_policy  : 'drp' (default), 'random' (A4), 'A_only', 'B_only', 'C_only'.
    space        : 'subspace' (paper: project on top-r) or 'full' (Mahalanobis under the shrunk
                   covariance in the whole space, residual lumped in one coordinate).
    split_at     : 'median' (paper) or 'mean' for Mode A.
    axis_sampling: 'argmax' (default, deterministic) or 'gain' (randomised DRP-AP-SA-R: the split
                   axis is sampled with probability proportional to gain**rand_gamma).
    rand_gamma   : sharpness of that sampling (default 4).
    n_random_dirs: extra candidate directions drawn at random in the top-q principal subspace
                   (weighted by sqrt(lambda)); only used with axis_sampling='gain'.
    rho          : optional extension (None = paper's rule). If set, Mode A is also used whenever
                   lambda_2/lambda_1 <= rho, i.e. the second direction is negligible.
    """

    def __init__(self, delta=0.05, kappa0=1e3, shrinkage="lw", eps_rel=1e-3, local_iters=5,
                 adaptive_k=False, metric="mahalanobis", mode_policy="drp", space="subspace",
                 split_at="median", dip_alpha=0.05, rho=None, max_axes=None, random_state=None,
                 axis_sampling="argmax", rand_gamma=4.0, n_random_dirs=0):
        self.axis_sampling = axis_sampling
        self.rand_gamma = rand_gamma
        self.n_random_dirs = n_random_dirs
        self.rho = rho
        self.max_axes = max_axes
        self.delta = delta
        self.kappa0 = kappa0
        self.shrinkage = shrinkage
        self.eps_rel = eps_rel
        self.local_iters = local_iters
        self.adaptive_k = adaptive_k
        self.metric = metric
        self.mode_policy = mode_policy
        self.space = space
        self.split_at = split_at
        self.dip_alpha = dip_alpha
        self.random_state = random_state

    # -- public API ---------------------------------------------------------
    def fit_init(self, X: np.ndarray, K: int) -> np.ndarray:
        X = np.asarray(X, dtype=float)
        n = len(X)
        rng = np.random.default_rng(self.random_state)
        self.trace_ = []
        Xc = X - X.mean(0)
        clusters = [(np.arange(n), centered_sse(Xc))]
        while len(clusters) < K:
            cand = [(i, c) for i, c in enumerate(clusters) if len(c[0]) >= 2 and c[1] > 1e-12]
            if not cand:
                break
            i = max(cand, key=lambda t: t[1][1])[0]
            idx, _ = clusters.pop(i)
            parts = self._split(X, idx, max_parts=K - len(clusters), rng=rng)
            for p in parts:
                sub = X[idx[p]]
                clusters.append((idx[p], centered_sse(sub - sub.mean(0))))
        cents = np.stack([X[c[0]].mean(0) for c in clusters])
        if len(cents) < K:  # degenerate data (fewer distinct points than K)
            extra = X[rng.choice(n, K - len(cents), replace=n < K - len(cents))]
            cents = np.vstack([cents, extra])
        return cents

    @property
    def mode_counts_(self):
        out = {"A": 0, "B": 0, "C": 0}
        for t in self.trace_:
            out[t["mode"]] = out.get(t["mode"], 0) + 1
        return out

    # -- internals ----------------------------------------------------------
    def _choose_mode(self, r, kappa, rng, ratio=0.0):
        p = self.mode_policy
        if p == "A_only":
            return "A"
        if p == "B_only":
            return "B"
        if p == "C_only":
            return "C"
        if p == "random":
            return str(rng.choice(["A", "B", "C"]))
        if r == 1 or (self.rho is not None and ratio <= self.rho):
            return "A"          # rho-gate (optional extension): negligible 2nd direction
        return "B" if kappa <= self.kappa0 else "C"

    def _split_arbitrate(self, Xc, Vt, r, rng=None):
        """SSE-arbitrated split (prototype 'DRP-AP-SA', deterministic).

        Candidates = median splits along u_1..u_q with q = min(max_axes or r_delta, rank).
        The candidate with the largest SSE reduction (between-group SS, exact in the full space)
        wins; u_1 is always a candidate, so each split removes at least as much SSE as PCA-Part's.
        The winner is then refined by `local_iters` Euclidean 2-means (Lloyd) iterations, which can
        only lower the split SSE.  No whitening, no randomness."""
        m = Xc.shape[0]
        q = min(self.max_axes if self.max_axes else r, Vt.shape[0])
        P = Xc @ Vt[:q].T
        dirs = [P[:, j] for j in range(q)]
        if self.axis_sampling == "gain" and self.n_random_dirs > 0 and rng is not None:
            lam_q = np.maximum((P ** 2).mean(0), 1e-300)
            for _ in range(self.n_random_dirs):
                w = rng.standard_normal(q) * np.sqrt(lam_q)
                w /= np.linalg.norm(w) + 1e-300
                dirs.append(P @ w)
        cands, gains = [], []
        for pj in dirs:
            L, R = median_split(pj, self.split_at)
            if len(L) == 0 or len(R) == 0:
                continue
            gains.append(len(L) * len(R) / m * float(((Xc[L].mean(0) - Xc[R].mean(0)) ** 2).sum()))
            cands.append((L, R))
        if not cands:
            L, R = median_split(Xc @ Vt[0], self.split_at)
            self._axis = 0
            return [L, R]
        gains = np.asarray(gains)
        if self.axis_sampling == "gain" and rng is not None:
            w = (gains / max(gains.max(), 1e-300)) ** self.rand_gamma
            j_ = int(rng.choice(len(cands), p=w / w.sum()))
            L, R = cands[j_]
            self._axis = j_
        else:
            L, R = cands[int(np.argmax(gains))]
            self._axis = int(np.argmax(gains))
        if self.local_iters > 0:
            c = np.stack([Xc[L].mean(0), Xc[R].mean(0)])
            lab = lloyd_in_space(Xc, c, self.local_iters)
            if lab is not None:
                L, R = np.where(lab == 0)[0], np.where(lab == 1)[0]
        return [L, R]

    def _split(self, X, idx, max_parts, rng):
        sub = X[idx]
        Xc = sub - sub.mean(0)
        m, d = Xc.shape
        lam, Vt = spectrum(Xc)
        r = directional_rank(lam, self.delta)
        kappa = conditioning_index(lam, m, d)
        ratio = float(lam[1] / lam[0]) if len(lam) > 1 and lam[0] > 0 else 0.0
        if self.mode_policy == "arbitrate":
            parts = self._split_arbitrate(Xc, Vt, r, rng)
            self.trace_.append(dict(size=m, r=r, kappa=kappa, mode="S", k_c=2, axis=self._axis,
                                    ratio=ratio))
            return parts
        mode = self._choose_mode(r, kappa, rng, ratio)
        k_c = 2
        parts = None

        if mode != "A" and len(lam) >= 2:
            rr = min(max(r, 2), len(lam))
            lam_t = lam.copy()
            lam_null = 0.0
            if mode == "C" and self.shrinkage != "none":
                lbar = float(lam.sum()) / d
                if self.shrinkage == "eps":
                    lam_t = lam + self.eps_rel * lbar
                    lam_null = self.eps_rel * lbar
                else:
                    a = shrinkage_alpha(Xc, self.shrinkage)
                    lam_t = (1.0 - a) * lam + a * lbar
                    lam_null = a * lbar
            floor = max(1e-12 * float(lam_t[0]), 1e-300)
            if self.metric == "mahalanobis":
                if self.space == "full" and mode == "C":
                    proj = Xc @ Vt.T
                    Z = proj / np.sqrt(np.maximum(lam_t, floor))
                    if lam_null > 0 and Vt.shape[0] < d:
                        resid = np.sqrt(np.maximum((Xc ** 2).sum(1) - (proj ** 2).sum(1), 0.0))
                        Z = np.hstack([Z, (resid / np.sqrt(lam_null))[:, None]])
                else:
                    Z = (Xc @ Vt[:rr].T) / np.sqrt(np.maximum(lam_t[:rr], floor))
            else:
                Z = Xc @ Vt[:rr].T

            if self.adaptive_k:
                n_sig = sum(unimodality_pvalue(Xc @ Vt[j]) < self.dip_alpha for j in range(rr))
                k_c = int(min(max(1 + n_sig, 2), max_parts))
            k_c = max(2, min(k_c, max_parts, m))
            seeds = kpp_indices(Z, k_c, rng)
            labels = lloyd_in_space(Z, Z[seeds].copy(), self.local_iters)
            if labels is not None:
                parts = [np.where(labels == j)[0] for j in range(k_c)]
            else:
                mode, k_c = "A", 2  # fallback: robust PCA-Part split

        if parts is None:
            L, R = median_split(Xc @ Vt[0], self.split_at)
            parts = [L, R]
            k_c = 2
        self.trace_.append(dict(size=m, r=r, kappa=kappa, mode=mode, k_c=k_c))
        return parts


class DRPAPSA(DRPAP):
    """DRP-AP-SA: the recommended, deterministic SSE-arbitrated variant.

    Defaults follow the full benchmark (results4):
      * local_iters=5  - per-split Lloyd refinement. T=0 is statistically indistinguishable from
                         PCA-Part on real data; T=5 carries the real gains (20NG, Isolet, Iris,
                         Olivetti, synthetic multi-axial). Known cost: +3.6% on COIL-20
                         (Portfolio avoids it). Use local_iters=0 for the cheaper variant;
      * max_axes=3     - arbitrate among the top-3 principal axes only (identical SSE to the
                         uncapped version on all 11 datasets, much cheaper in high dimension).
    """

    def __init__(self, local_iters=5, max_axes=3, **kw):
        kw.pop("mode_policy", None)
        super().__init__(mode_policy="arbitrate", local_iters=local_iters, max_axes=max_axes, **kw)


class DRPAPSAR(DRPAPSA):
    """DRP-AP-SA-R: randomised SSE-arbitrated splitting, meant to be used inside a restart pool.

    The split axis is sampled with probability proportional to (gain / best gain)**rand_gamma over
    the top-q principal axes plus `n_random_dirs` random directions of the top-q subspace, so
    the best axis is the most likely but not the only one.  With a different seed each call gives a
    different, still SSE-informed, initialisation."""

    def __init__(self, rand_gamma=4.0, n_random_dirs=3, **kw):
        kw.pop("axis_sampling", None)
        super().__init__(axis_sampling="gain", rand_gamma=rand_gamma,
                         n_random_dirs=n_random_dirs, **kw)

In [ ]:
%%writefile drp_ap/baselines.py
"""Baseline k-means initializers (protocol of the paper).

Every function has signature  f(X, K, rng) -> centroids (K, d)  with rng a numpy Generator,
except deterministic ones which ignore rng.
"""
from __future__ import annotations

import numpy as np
from sklearn.cluster import KMeans, kmeans_plusplus
from sklearn.metrics import pairwise_distances_argmin_min

from .core import centered_sse, median_split, spectrum


def _seed(rng) -> int:
    return int(rng.integers(0, 2 ** 31 - 1))


# --- random / k-means++ family ------------------------------------------------

def init_random(X, K, rng):
    return X[rng.choice(len(X), K, replace=len(X) < K)].astype(float)


def init_kmeanspp(X, K, rng):
    return kmeans_plusplus(X, K, random_state=_seed(rng), n_local_trials=1)[0]


def init_greedy_kmeanspp(X, K, rng):
    """scikit-learn default: best of 2 + floor(log K) candidates per step."""
    return kmeans_plusplus(X, K, random_state=_seed(rng))[0]


# --- deterministic ------------------------------------------------------------

def init_kkz(X, K, rng=None):
    mu = X.mean(0)
    i0 = int(np.argmax(((X - mu) ** 2).sum(1)))
    chosen = [i0]
    dmin = ((X - X[i0]) ** 2).sum(1)
    for _ in range(1, K):
        i = int(np.argmax(dmin))
        chosen.append(i)
        dmin = np.minimum(dmin, ((X - X[i]) ** 2).sum(1))
    return X[chosen].astype(float)


def _lloyd2(sub, c0, T):
    """T Lloyd iterations of 2-means in the full space, starting from centres c0 (2,d)."""
    c = c0.copy()
    lab = None
    for _ in range(T):
        lab = ((sub[:, None, :] - c[None]) ** 2).sum(2).argmin(1)
        if lab.min() == lab.max():
            return None
        c = np.stack([sub[lab == 0].mean(0), sub[lab == 1].mean(0)])
    return lab


def _divisive(X, K, direction, split_at="median", local_iters=0):
    n = len(X)
    Xc0 = X - X.mean(0)
    clusters = [(np.arange(n), centered_sse(Xc0))]
    while len(clusters) < K:
        cand = [(i, c) for i, c in enumerate(clusters) if len(c[0]) >= 2 and c[1] > 1e-12]
        if not cand:
            break
        i = max(cand, key=lambda t: t[1][1])[0]
        idx, _ = clusters.pop(i)
        sub = X[idx]
        Xc = sub - sub.mean(0)
        proj = Xc @ direction(Xc)
        L, R = median_split(proj, split_at)
        if local_iters > 0:
            c0 = np.stack([sub[L].mean(0), sub[R].mean(0)])
            lab = _lloyd2(sub, c0, local_iters)
            if lab is not None:
                L, R = np.where(lab == 0)[0], np.where(lab == 1)[0]
        for p in (L, R):
            s = X[idx[p]]
            clusters.append((idx[p], centered_sse(s - s.mean(0))))
    cents = np.stack([X[c[0]].mean(0) for c in clusters])
    if len(cents) < K:
        cents = np.vstack([cents, X[np.random.default_rng(0).choice(n, K - len(cents))]])
    return cents


def init_pca_part(X, K, rng=None, split_at="median", local_iters=0):
    """PCA-Part (Su & Dy 2004): split the largest-SSE cluster along its first PC."""
    return _divisive(X, K, lambda Xc: spectrum(Xc)[1][0], split_at, local_iters)


def init_var_part(X, K, rng=None, split_at="median"):
    """Var-Part (Su & Dy 2007): split along the coordinate of largest variance."""
    def direction(Xc):
        e = np.zeros(Xc.shape[1])
        e[int(np.argmax(Xc.var(0)))] = 1.0
        return e
    return _divisive(X, K, direction, split_at)


# --- Bradley & Fayyad refinement ---------------------------------------------

def init_bradley_fayyad(X, K, rng, J=10, frac=0.1):
    n = len(X)
    m = int(min(n, max(frac * n, 5 * K)))
    sols = []
    for _ in range(J):
        sub = X[rng.choice(n, m, replace=False)]
        km = KMeans(K, init="random", n_init=1, max_iter=50, random_state=_seed(rng)).fit(sub)
        sols.append(km.cluster_centers_)
    CM = np.vstack(sols)
    best, best_in = None, np.inf
    for s in sols:
        km = KMeans(K, init=s, n_init=1, max_iter=50).fit(CM)
        if km.inertia_ < best_in:
            best, best_in = km.cluster_centers_, km.inertia_
    return best


# --- Mahalanobis seeding (K-means++ MD and variants) ---------------------------

def whitening(X, eps_rel=1e-6):
    """Global whitening W so that Euclidean distance in (X-mu) @ W is the Mahalanobis distance.

    The ad hoc isotropic term eps*I is the standard patch for singular covariances.
    """
    mu = X.mean(0)
    C = np.cov(X, rowvar=False, bias=True)
    C = np.atleast_2d(C)
    lbar = float(np.trace(C)) / C.shape[0]
    w, V = np.linalg.eigh(C + eps_rel * lbar * np.eye(C.shape[0]))
    w = np.maximum(w, 1e-300)
    return mu, V / np.sqrt(w)


def init_kmeanspp_md(X, K, rng, eps_rel=1e-6):
    mu, W = whitening(X, eps_rel)
    Z = (X - mu) @ W
    idx = kmeans_plusplus(Z, K, random_state=_seed(rng), n_local_trials=1)[1]
    return X[idx].astype(float)


def init_kmeanspp_md_best(X, K, rng, R=10, eps_rel=1e-6):
    """Best of R Mahalanobis seedings according to Mahalanobis inertia."""
    mu, W = whitening(X, eps_rel)
    Z = (X - mu) @ W
    best, best_in = None, np.inf
    for _ in range(R):
        idx = kmeans_plusplus(Z, K, random_state=_seed(rng), n_local_trials=1)[1]
        _, dmin = pairwise_distances_argmin_min(Z, Z[idx])
        inertia = float((dmin ** 2).sum())
        if inertia < best_in:
            best, best_in = idx, inertia
    return X[best].astype(float)


def init_kmeanspp_md_hybrid(X, K, rng, eps_rel=1e-6):
    """Hybrid metric: D^2 sampling under the average of the (trace-normalised) Euclidean and
    Mahalanobis squared distances.  NOTE: this is our operationalisation of 'Hybrid' - adapt it
    if the definition in the reference differs."""
    mu, W = whitening(X, eps_rel)
    Zm = (X - mu) @ W
    tr = float(((X - mu) ** 2).sum(1).mean())
    d = X.shape[1]
    Zh = np.hstack([(X - mu) / np.sqrt(2 * max(tr, 1e-300)), Zm / np.sqrt(2 * d)])
    idx = kmeans_plusplus(Zh, K, random_state=_seed(rng), n_local_trials=1)[1]
    return X[idx].astype(float)


# --- Model-based reference: GMM means as initial centroids ---------------------

def init_gmm(X, K, rng, max_iter=20):
    import warnings

    from sklearn.exceptions import ConvergenceWarning
    from sklearn.mixture import GaussianMixture

    for reg in (1e-6, 1e-3, 1e-1):
        try:
            with warnings.catch_warnings():       # EM is deliberately capped (max_iter)
                warnings.simplefilter("ignore", ConvergenceWarning)
                g = GaussianMixture(K, covariance_type="full", init_params="k-means++",
                                    reg_covar=reg * float(X.var(0).mean()), max_iter=max_iter,
                                    random_state=_seed(rng)).fit(X)
            return g.means_
        except Exception:
            continue
    return init_kmeanspp(X, K, rng)

In [ ]:
%%writefile drp_ap/metrics.py
"""Metrics of the protocol: SSE, ARI, NMI, silhouette, seed stability, failure rate."""
from __future__ import annotations

import numpy as np
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             pairwise_distances_argmin_min, silhouette_score)


def sse(X, C) -> float:
    _, d = pairwise_distances_argmin_min(X, C)
    return float((d ** 2).sum())


def external_scores(y, labels):
    if y is None:
        return np.nan, np.nan
    return (float(adjusted_rand_score(y, labels)),
            float(normalized_mutual_info_score(y, labels)))


def silhouette(X, labels, sample=2000, seed=0) -> float:
    if len(np.unique(labels)) < 2:
        return np.nan
    return float(silhouette_score(X, labels, sample_size=min(sample, len(X) - 1), random_state=seed)
                 if len(X) > sample else silhouette_score(X, labels))


def diameter(X) -> float:
    """Bounding-box diagonal (cheap upper bound of the data diameter)."""
    return float(np.linalg.norm(X.max(0) - X.min(0)))


def matched_distance(A, B) -> float:
    """Mean Euclidean distance between two centroid sets after optimal (Hungarian) matching."""
    D = cdist(A, B)
    r, c = linear_sum_assignment(D)
    return float(D[r, c].mean())


def seed_stability(centroid_sets, diam) -> float:
    """Average pairwise matched distance between initial centroids across resamples,
    normalised by the data diameter (lower = more stable)."""
    B = len(centroid_sets)
    vals = [matched_distance(centroid_sets[i], centroid_sets[j])
            for i in range(B) for j in range(i + 1, B)]
    return float(np.mean(vals) / max(diam, 1e-300)) if vals else np.nan

In [ ]:
%%writefile drp_ap/stats.py
"""Non-parametric statistical testing (Demsar 2006): Friedman + Nemenyi, Wilcoxon + Holm."""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats


def average_ranks(pivot: pd.DataFrame) -> pd.Series:
    """pivot: datasets x methods of a 'lower is better' score -> mean rank per method."""
    return pivot.rank(axis=1, method="average").mean(0).sort_values()


def friedman(pivot: pd.DataFrame):
    if pivot.shape[0] < 3 or pivot.shape[1] < 3:
        return np.nan, np.nan
    stat, p = stats.friedmanchisquare(*[pivot[c].values for c in pivot.columns])
    return float(stat), float(p)


def nemenyi_cd(k: int, n: int, alpha: float = 0.05) -> float:
    """Critical difference CD = q_alpha * sqrt(k(k+1)/(6N)) with q from the studentized range."""
    q = stats.studentized_range.ppf(1 - alpha, k, np.inf) / np.sqrt(2)
    return float(q * np.sqrt(k * (k + 1) / (6.0 * n)))


def holm(pvals: dict) -> dict:
    items = sorted(pvals.items(), key=lambda t: t[1])
    m = len(items)
    adj, running = {}, 0.0
    for i, (k, p) in enumerate(items):
        running = max(running, min(1.0, (m - i) * p))
        adj[k] = running
    return adj


def wilcoxon_vs_ref(pivot: pd.DataFrame, ref: str, alternative: str = "two-sided") -> pd.DataFrame:
    """Paired Wilcoxon of every method vs `ref` across rows, Holm-corrected."""
    raw, eff = {}, {}
    for c in pivot.columns:
        if c == ref:
            continue
        diff = pivot[c].values - pivot[ref].values
        eff[c] = float(np.median(diff))
        if np.allclose(diff, 0):
            raw[c] = 1.0
            continue
        try:
            raw[c] = float(stats.wilcoxon(pivot[c].values, pivot[ref].values,
                                          alternative=alternative).pvalue)
        except ValueError:
            raw[c] = 1.0
    adj = holm(raw)
    return pd.DataFrame({"median_diff_vs_ref": eff, "p": raw, "p_holm": adj})


def cd_diagram(ranks: pd.Series, cd: float, path: str, title: str = ""):
    """Compact critical-difference diagram (methods within CD of each other are joined)."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    names, vals = list(ranks.index), ranks.values
    k = len(vals)
    fig, ax = plt.subplots(figsize=(9, 1.2 + 0.28 * k))
    lo, hi = 1, max(k, vals.max())
    ax.set_xlim(lo - 0.3, hi + 0.3)
    ax.set_ylim(-k - 2.2, 1.2)
    ax.hlines(0, lo, hi, color="k", lw=1)
    for t in range(int(lo), int(hi) + 1):
        ax.vlines(t, 0, 0.12, color="k", lw=1)
        ax.text(t, 0.25, str(t), ha="center", fontsize=8)
    ax.hlines(0.9, lo, lo + cd, color="C3", lw=2)
    ax.text(lo + cd / 2, 1.0, f"CD = {cd:.2f}", ha="center", fontsize=8, color="C3")
    for i, (n, v) in enumerate(zip(names, vals)):
        y = -1 - i
        ax.hlines(y, lo - 0.3, v, color="0.6", lw=0.7)
        ax.vlines(v, y, 0, color="0.6", lw=0.7)
        ax.text(lo - 0.32, y, f"{n}  ({v:.2f})", ha="right", va="center", fontsize=8)
    y0 = -k - 0.8
    row = 0
    i = 0
    while i < k:  # maximal cliques of consecutive methods
        j = i
        while j + 1 < k and vals[j + 1] - vals[i] <= cd:
            j += 1
        if j > i:
            ax.hlines(y0 - 0.25 * row, vals[i], vals[j], color="k", lw=3)
            row += 1
        i = j + 1 if j > i else i + 1
    ax.axis("off")
    ax.set_title(title, fontsize=10)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)

In [ ]:
%%writefile drp_ap/synthetic.py
"""Synthetic generators with controlled directional rank (Section 'Synthetic generators').

All generators return (X, y, K) and apply a random rotation so that no method benefits from
axis-aligned structure (Var-Part would otherwise be advantaged).
"""
from __future__ import annotations

import numpy as np


def _rotation(d, rng):
    Q, R = np.linalg.qr(rng.standard_normal((d, d)))
    return Q * np.sign(np.diag(R))


def _assign(n, K, rng):
    y = np.arange(n) % K
    rng.shuffle(y)
    return y


def mono_directional(n=1500, d=10, K=4, sep=4.0, rng=None, seed=0):
    """Clusters separated along a single direction (regime of Theorem 1: r_delta = 1)."""
    rng = rng or np.random.default_rng(seed)
    y = _assign(n, K, rng)
    means = np.zeros((K, d))
    means[:, 0] = sep * (np.arange(K) - (K - 1) / 2) * 1.0
    X = means[y] + rng.standard_normal((n, d)) * 0.5
    return X @ _rotation(d, rng), y, K


def multi_axial(n=2000, d=20, r=2, ratio=0.6, sigma1=1.0, sigma2=0.3, noise=0.05,
                rng=None, seed=0):
    """Theorem-2 construction: K = 2^r clusters at hypercube vertices +-a on axes 2..r+1, each
    cluster elongated along axis 1.  Total spectrum: lam_1 = sigma1^2,
    lam_2..lam_{r+1} = sigma2^2 + a^2 = ratio * sigma1^2  (so lambda_2/lambda_1 = ratio).
    Remaining d-r-1 axes carry tiny variance `noise`^2 (ill-conditioning)."""
    rng = rng or np.random.default_rng(seed)
    d = max(d, r + 2)
    a = np.sqrt(max(ratio * sigma1 ** 2 - sigma2 ** 2, 1e-6))
    K = 2 ** r
    y = _assign(n, K, rng)
    signs = np.array([[1 if (k >> j) & 1 else -1 for j in range(r)] for k in range(K)], float)
    means = np.zeros((K, d))
    means[:, 1:r + 1] = a * signs
    std = np.full(d, noise)
    std[0] = sigma1
    std[1:r + 1] = sigma2
    X = means[y] + rng.standard_normal((n, d)) * std
    return X @ _rotation(d, rng), y, K


def anisotropic(n=2000, d=20, K=5, kappa=1e4, spread=6.0, rng=None, seed=0):
    """Gaussian clusters whose covariances have eigenvalues log-spaced in [1/kappa, 1],
    each with its own random orientation (ill-conditioned Mahalanobis metric)."""
    rng = rng or np.random.default_rng(seed)
    y = _assign(n, K, rng)
    ev = np.logspace(0, -np.log10(kappa), d)
    centers = rng.standard_normal((K, d)) * spread / np.sqrt(d) * 2
    X = np.empty((n, d))
    for k in range(K):
        sel = y == k
        Q = _rotation(d, rng)
        X[sel] = centers[k] + (rng.standard_normal((sel.sum(), d)) * np.sqrt(ev)) @ Q.T
    return X, y, K


def concentric_shells(n=1500, d=2, K=3, noise=0.05, rng=None, seed=0):
    """Concentric shells (sanity check; SSE-based methods are not expected to recover them)."""
    rng = rng or np.random.default_rng(seed)
    y = _assign(n, K, rng)
    g = rng.standard_normal((n, d))
    g /= np.linalg.norm(g, axis=1, keepdims=True)
    X = g * (1 + 2 * y)[:, None] + rng.standard_normal((n, d)) * noise
    return X, y, K


def high_dim_mixture(n=200, d=50, K=5, sep=3.0, rng=None, seed=0):
    """High-dimensional mixture with n comparable to d (small-sample conditioning regime):
    low-rank cluster covariances plus isotropic noise."""
    rng = rng or np.random.default_rng(seed)
    y = _assign(n, K, rng)
    centers = rng.standard_normal((K, d)) * sep / np.sqrt(d) * np.sqrt(d) / 3
    X = np.empty((n, d))
    for k in range(K):
        sel = y == k
        B = rng.standard_normal((d, 3)) * 1.5
        X[sel] = (centers[k] + rng.standard_normal((sel.sum(), 3)) @ B.T
                  + rng.standard_normal((sel.sum(), d)) * 0.3)
    return X, y, K


# ---------------------------------------------------------------------------
# Generators that are NOT the Theorem-2 construction (non-circular evaluation)
# ---------------------------------------------------------------------------

def subspace_gmm(n=2000, d=20, K=8, r=3, decay=0.7, noise=0.5, rng=None, seed=0):
    """Gaussian clusters whose centres are random in an r-dim subspace with geometrically decaying
    between-cluster spread (decay=1: equal spread on all r axes => high directional rank).
    No hypercube structure, random K, isotropic within-cluster noise."""
    rng = rng or np.random.default_rng(seed)
    d = max(d, r + 1)
    y = _assign(n, K, rng)
    scale = decay ** np.arange(r)
    centers = np.zeros((K, d))
    centers[:, :r] = rng.standard_normal((K, r)) * scale * 3.0
    X = centers[y] + rng.standard_normal((n, d)) * noise
    return X @ _rotation(d, rng), y, K


def random_gmm(n=2000, d=20, K=6, overlap=1.0, rng=None, seed=0):
    """Random-covariance Gaussian mixture: every cluster has its own random rotation, random
    spectrum decay exponent and random scale; centres are N(0, I) scaled by 1/overlap."""
    rng = rng or np.random.default_rng(seed)
    y = _assign(n, K, rng)
    centers = rng.standard_normal((K, d)) * 2.0 / overlap
    X = np.empty((n, d))
    for k in range(K):
        sel = y == k
        expo = rng.uniform(0.3, 2.0)
        ev = (np.arange(1, d + 1) ** -expo) * rng.uniform(0.5, 2.0)
        Q = _rotation(d, rng)
        X[sel] = centers[k] + (rng.standard_normal((sel.sum(), d)) * np.sqrt(ev)) @ Q.T
    return X, y, K


def student_mixture(n=2000, d=20, K=6, df=4, rng=None, seed=0):
    """Heavy-tailed (multivariate Student-t) clusters with random centres and spectra."""
    rng = rng or np.random.default_rng(seed)
    y = _assign(n, K, rng)
    centers = rng.standard_normal((K, d)) * 2.5
    X = np.empty((n, d))
    for k in range(K):
        sel = y == k
        m = int(sel.sum())
        ev = np.sort(rng.uniform(0.05, 1.0, d))[::-1]
        g = rng.standard_normal((m, d)) * np.sqrt(ev) / np.sqrt(rng.chisquare(df, (m, 1)) / df)
        X[sel] = centers[k] + g @ _rotation(d, rng).T
    return X, y, K


def ring_mixture(n=2000, d=20, K=8, radius=3.0, noise=0.6, rng=None, seed=0):
    """K clusters placed on a circle in a random 2-D plane: lambda_1 ~ lambda_2 (maximal 2-axis
    ambiguity for PCA-Part) without any hypercube or elongation structure."""
    rng = rng or np.random.default_rng(seed)
    y = _assign(n, K, rng)
    ang = 2 * np.pi * (np.arange(K) + rng.uniform(0, 1)) / K
    centers = np.zeros((K, d))
    centers[:, 0], centers[:, 1] = radius * np.cos(ang), radius * np.sin(ang)
    X = centers[y] + rng.standard_normal((n, d)) * noise
    return X @ _rotation(d, rng), y, K


FAMILIES = {
    "mono": mono_directional,
    "multi_axial": multi_axial,
    "anisotropic": anisotropic,
    "shells": concentric_shells,
    "highdim": high_dim_mixture,
    "subspace_gmm": subspace_gmm,
    "random_gmm": random_gmm,
    "student": student_mixture,
    "ring": ring_mixture,
}

In [ ]:
%%writefile drp_ap/data.py
"""The twelve benchmark datasets of Table 1 (tabular, image, text, audio, sensor).

Bundled with scikit-learn (no download): iris, wine, breast_cancer, digits.
Downloaded on first use (internet needed, fine on Colab): MNIST, Fashion-MNIST, COIL-20,
Olivetti, 20 Newsgroups, Reuters R8, Isolet, UCI HAR.

Any dataset that cannot be obtained is skipped with a warning; you can also drop a file
`<data_dir>/<name>.npz` containing arrays X and y to supply it manually.
"""
from __future__ import annotations

import io
import os
import urllib.request
import warnings
import zipfile

import numpy as np
from sklearn import datasets as skd
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.preprocessing import Normalizer, StandardScaler

DEFAULT_DIR = os.environ.get("DRP_AP_DATA", "./data_cache")

# name -> (modality, description)
DATASETS = {
    "Iris": "tabular", "Wine": "tabular", "BC": "tabular",
    "Digits": "image", "MNIST": "image", "F-MNIST": "image", "COIL-20": "image",
    "Olivetti": "image", "20NG": "text", "R8": "text", "Isolet": "audio", "HAR": "sensor",
}


# Extended protocol: 18 additional datasets from OpenML (downloaded on first use;
# the names are the OpenML dataset names and were NOT checked offline: a failing name is skipped with
# a warning, drop <data_dir>/<name>.npz to supply it). Together with the 12 core ones: the 30-dataset benchmark.
EXTENDED = {
    "glass": "tabular", "ecoli": "tabular", "yeast": "tabular", "vehicle": "tabular",
    "segment": "image-features", "satimage": "remote-sensing", "pendigits": "pen-trajectory",
    "letter": "image-features", "waveform-5000": "signal", "spambase": "text-features",
    "ionosphere": "signal", "sonar": "signal", "vowel": "audio", "page-blocks": "document",
    "mfeat-karhunen": "image-features", "mfeat-fourier": "image-features", "optdigits": "image",
    "dermatology": "tabular",
}
ALL_NAMES = list(DATASETS) + list(EXTENDED)

# Confirmation set: chosen BEFORE running the confirmation experiment, disjoint from the 30 above.
CONFIRM = {
    "mfeat-zernike": "tabular", "mfeat-factors": "tabular", "mfeat-pixel": "tabular",
    "mfeat-morphological": "tabular", "texture": "tabular", "wall-robot-navigation": "sensor",
    "cardiotocography": "tabular", "steel-plates-fault": "tabular",
    "one-hundred-plants-margin": "tabular", "one-hundred-plants-texture": "tabular",
    "one-hundred-plants-shape": "tabular", "phoneme": "audio", "balance-scale": "tabular",
    "JapaneseVowels": "audio", "collins": "text", "cnae-9": "text", "semeion": "image",
}


def _openml_tab(name, data_dir):
    """OpenML tabular dataset: one-hot for categorical columns, median imputation, z-scoring, PCA to
    <=50 dimensions when d is larger."""
    import pandas as pd

    last = None
    for ver in (1, "active"):                      # version 1 first, then the active version
        for _ in range(2):                         # one retry on transient network errors
            try:
                b = skd.fetch_openml(name, version=ver, as_frame=True, data_home=data_dir)
                break
            except Exception as e:                 # noqa: BLE001
                last = e
        else:
            continue
        break
    else:
        raise last
    X = pd.get_dummies(b.data, dtype=float)
    X = X.fillna(X.median()).to_numpy(float)
    X = StandardScaler().fit_transform(X)
    X = np.nan_to_num(X)
    if X.shape[1] > 50:
        X = _pca(X)
    return X, _factorize(b.target)


def _pca(X, d=50, seed=0):
    d = min(d, X.shape[1], X.shape[0] - 1)
    return PCA(d, random_state=seed).fit_transform(X)


def _factorize(y):
    return np.unique(np.asarray(y), return_inverse=True)[1]


def _openml(name, data_dir, version=1):
    b = skd.fetch_openml(name, version=version, as_frame=False, data_home=data_dir)
    return np.asarray(b.data, dtype=float), _factorize(b.target)


def _tfidf_lsa(texts, n_comp=100, seed=0):
    from sklearn.feature_extraction.text import TfidfVectorizer

    V = TfidfVectorizer(max_features=20000, stop_words="english", min_df=5, sublinear_tf=True)
    T = V.fit_transform(texts)
    Z = TruncatedSVD(min(n_comp, T.shape[1] - 1), random_state=seed).fit_transform(T)
    return Normalizer().fit_transform(Z)


def _load_coil20(data_dir):
    url = ("https://www.cs.columbia.edu/CAVE/databases/SLAM_coil-20_coil-100/coil-20/"
           "coil-20-proc.zip")
    from PIL import Image

    raw = urllib.request.urlopen(url, timeout=60).read()
    X, y = [], []
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        for nm in sorted(z.namelist()):
            if nm.lower().endswith(".png"):
                lab = int(os.path.basename(nm).split("__")[0].replace("obj", ""))
                im = Image.open(z.open(nm)).convert("L").resize((32, 32))
                X.append(np.asarray(im, dtype=float).ravel() / 255.0)
                y.append(lab)
    return np.array(X), _factorize(y)


def _load_r8(data_dir):
    base = "https://www.cs.umb.edu/~smimarog/textmining/datasets/"
    texts, labels = [], []
    for f in ("r8-train-no-stop.txt", "r8-test-no-stop.txt"):
        local = os.path.join(data_dir, f)          # manual copy of the raw file takes priority
        if os.path.exists(local):
            raw = open(local, encoding="utf-8", errors="ignore").read()
        else:
            raw = urllib.request.urlopen(base + f, timeout=60).read().decode("utf-8", "ignore")
        for line in raw.splitlines():
            if "\t" in line:
                lab, txt = line.split("\t", 1)
                labels.append(lab)
                texts.append(txt)
    return _tfidf_lsa(texts), _factorize(labels)


def _raw(name, data_dir):
    if name == "Iris":
        b = skd.load_iris(); return StandardScaler().fit_transform(b.data), b.target
    if name == "Wine":
        b = skd.load_wine(); return StandardScaler().fit_transform(b.data), b.target
    if name == "BC":
        b = skd.load_breast_cancer(); return StandardScaler().fit_transform(b.data), b.target
    if name == "Digits":
        b = skd.load_digits(); return b.data.astype(float), b.target
    if name == "MNIST":
        X, y = _openml("mnist_784", data_dir); return _pca(X / 255.0), y
    if name == "F-MNIST":
        X, y = _openml("Fashion-MNIST", data_dir); return _pca(X / 255.0), y
    if name == "COIL-20":
        X, y = _load_coil20(data_dir); return _pca(X), y
    if name == "Olivetti":
        b = skd.fetch_olivetti_faces(data_home=data_dir); return _pca(b.data), b.target
    if name == "20NG":
        b = skd.fetch_20newsgroups(subset="all", remove=("headers", "footers", "quotes"),
                                   data_home=data_dir)
        return _tfidf_lsa(b.data), b.target
    if name == "R8":
        return _load_r8(data_dir)
    if name == "Isolet":
        X, y = _openml("isolet", data_dir); return _pca(StandardScaler().fit_transform(X)), y
    if name == "HAR":
        X, y = _openml("har", data_dir); return _pca(StandardScaler().fit_transform(X)), y
    if name in EXTENDED or name in CONFIRM:
        return _openml_tab(name, data_dir)
    raise KeyError(name)


def load_dataset(name, data_dir=DEFAULT_DIR, max_n=None, seed=0):
    """Return (X, y, K). `max_n` draws a stratified subsample (for speed on Colab)."""
    os.makedirs(data_dir, exist_ok=True)
    manual = os.path.join(data_dir, f"{name}.npz")
    if os.path.exists(manual):
        z = np.load(manual, allow_pickle=True)
        X, y = z["X"].astype(float), _factorize(z["y"])
    else:
        X, y = _raw(name, data_dir)
    X = np.asarray(X, dtype=float)
    y = _factorize(y)
    if max_n and len(X) > max_n:
        rng = np.random.default_rng(seed)
        keep = []
        for c in np.unique(y):
            ids = np.where(y == c)[0]
            keep.append(rng.choice(ids, max(int(round(max_n * len(ids) / len(X))), 2),
                                   replace=False))
        keep = np.sort(np.concatenate(keep))
        X, y = X[keep], y[keep]
    return X, y, int(len(np.unique(y)))


def manifest(data):
    """One row per loaded dataset: modality, n, d, K, smallest class size (for the paper's table)."""
    import pandas as pd
    mod = {**DATASETS, **EXTENDED, **CONFIRM}
    rows = []
    for n, (X, y, K) in data.items():
        rows.append(dict(dataset=n, modality=mod.get(n, "?"), n=len(X), d=X.shape[1], K=K,
                         min_class=int(np.bincount(y).min()), n_per_class=round(len(X) / K, 1)))
    return pd.DataFrame(rows)


def load_many(names, data_dir=DEFAULT_DIR, max_n=None, seed=0, verbose=True):
    out = {}
    for n in names:
        try:
            out[n] = load_dataset(n, data_dir, max_n, seed)
            if verbose:
                X, y, K = out[n]
                print(f"[data] {n:9s} n={X.shape[0]:6d} d={X.shape[1]:4d} K={K}")
        except Exception as e:  # network failure, missing file...
            warnings.warn(f"Dataset {n} skipped: {type(e).__name__}: {str(e)[:120]}. "
                          f"Provide {os.path.join(data_dir, n + '.npz')} (arrays X, y) to add it.")
    return out

In [ ]:
%%writefile drp_ap/experiments.py
"""Experiment runners implementing the protocol of the paper."""
from __future__ import annotations

import itertools
import time
import warnings
from functools import partial

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

from . import baselines as B
from . import metrics as M
from .core import DRPAP, DRPAPSA, DRPAPSAR
from .synthetic import FAMILIES

TOL, MAX_ITER = 1e-4, 300


# ----------------------------------------------------------------------------
# Method registry:  name -> (callable(X, K, seed) -> centroids, deterministic?)
# ----------------------------------------------------------------------------

def _rngf(f):
    return lambda X, K, seed: f(X, K, np.random.default_rng(seed))


def _drp(**kw):
    return lambda X, K, seed: DRPAP(random_state=seed, **kw).fit_init(X, K)


def _sa(**kw):
    """DRP-AP-SA with the recommended defaults (T=5, max_axes=3) unless overridden."""
    return lambda X, K, seed: DRPAPSA(random_state=seed, **kw).fit_init(X, K)


def _final_sse(X, K, C):
    return KMeans(K, init=C, n_init=1, max_iter=MAX_ITER, tol=TOL, algorithm="lloyd",
                  random_state=0).fit(X).inertia_


def best_of(f, R):
    """Equal-compute restart baseline: R independent (init + Lloyd) runs of the stochastic
    initialiser f(X, K, rng); returns the initial centroids with the lowest final SSE.
    The reported init time therefore includes the R Lloyd runs."""
    def run(X, K, seed):
        rng = np.random.default_rng(seed)
        best, best_in = None, np.inf
        for _ in range(R):
            C = np.asarray(f(X, K, rng), dtype=float)
            v = _final_sse(X, K, C)
            if v < best_in:
                best, best_in = C, v
        return best
    return run


def best_of_pool(anchor, f_rand, R):
    """Pool of R Lloyd runs: the deterministic `anchor(X, K)` first, then R-1 draws of the
    randomised initialiser f_rand(X, K, rng); keep the lowest final SSE (equal compute to best_of(R))."""
    def run(X, K, seed):
        rng = np.random.default_rng(seed)
        best = np.asarray(anchor(X, K), dtype=float)
        best_in = _final_sse(X, K, best)
        for _ in range(R - 1):
            C = np.asarray(f_rand(X, K, rng), dtype=float)
            v = _final_sse(X, K, C)
            if v < best_in:
                best, best_in = C, v
        return best
    return run


def _sar(X, K, rng):
    return DRPAPSAR(random_state=int(rng.integers(2 ** 31 - 1))).fit_init(X, K)


def init_portfolio(X, K, seed=0):
    """Run Lloyd from several deterministic initialisations; return the initial centroids whose
    final SSE is lowest (selection uses only the unsupervised objective)."""
    cands = [B.init_pca_part(X, K), B.init_var_part(X, K),
             DRPAPSA(random_state=seed).fit_init(X, K)]   # SA (T=5, q<=3)
    best, best_in = None, np.inf
    for C in cands:
        v = _final_sse(X, K, C)
        if v < best_in:
            best, best_in = C, v
    return best


def main_methods():
    return {
        "Random": (_rngf(B.init_random), False),
        "k-means++": (_rngf(B.init_kmeanspp), False),
        "Greedy k-means++": (_rngf(B.init_greedy_kmeanspp), False),
        "KKZ": (lambda X, K, s: B.init_kkz(X, K), True),
        "PCA-Part": (lambda X, K, s: B.init_pca_part(X, K), True),
        "Var-Part": (lambda X, K, s: B.init_var_part(X, K), True),
        "Bradley-Fayyad": (_rngf(B.init_bradley_fayyad), False),
        "k-means++ MD": (_rngf(B.init_kmeanspp_md), False),
        "k-means++ MD (best of 10)": (_rngf(B.init_kmeanspp_md_best), False),
        "k-means++ MD Hybrid": (_rngf(B.init_kmeanspp_md_hybrid), False),
        "GMM-init": (_rngf(B.init_gmm), False),
        "DRP-AP": (_drp(), False),
        "DRP-AP+": (_drp(adaptive_k=True), False),
        # recommended variant: deterministic SSE-arbitrated splits, per-split Lloyd T=5, top-3 axes
        "DRP-AP-SA": (_sa(), True),
        # faithful Su & Dy baselines: the original PCA-Part / Var-Part split at the MEAN of the
        # projection (hyperplane through the centroid); "PCA-Part"/"Var-Part" above split at the median
        "PCA-Part (Su-Dy, mean split)": (lambda X, K, s: B.init_pca_part(X, K, split_at="mean"), True),
        "Var-Part (Su-Dy, mean split)": (lambda X, K, s: B.init_var_part(X, K, split_at="mean"), True),
        "DRP-AP-SA (mean split)": (_sa(split_at="mean"), True),
        # cheaper variant without per-split refinement (== PCA-Part on real data in results4)
        "DRP-AP-SA (T=0)": (_sa(local_iters=0), True),
        # A3 as a first-class baseline: PCA-Part with the same local refinement
        "PCA-Part + local Lloyd (A3)": (lambda X, K, s: B.init_pca_part(X, K, local_iters=5), True),
        # equal-compute restart baselines (R full init+Lloyd runs, keep the best)
        "Greedy k-means++ best-of-3": (best_of(B.init_greedy_kmeanspp, 3), False),
        "Greedy k-means++ best-of-10": (best_of(B.init_greedy_kmeanspp, 10), False),
        # randomised DRP-AP-SA-R restarts and a pool anchored on the deterministic DRP-AP-SA
        "SA-R best-of-10": (best_of(_sar, 10), False),
        "SA pool-3": (best_of_pool(lambda X, K: DRPAPSA().fit_init(X, K), _sar, 3), False),
        "SA pool-10": (best_of_pool(lambda X, K: DRPAPSA().fit_init(X, K), _sar, 10), False),
        # deterministic portfolio: Lloyd from {PCA-Part, Var-Part, SA(T=5)}, keep the lowest final SSE
        # (costs 3 Lloyd runs; the init time reported includes them)
        "Portfolio {PCA,Var,SA}": (lambda X, K, s: init_portfolio(X, K, s), True),
    }


def ablation_methods():
    return {
        "DRP-AP (full)": (_drp(), False),
        "A1: delta=1 (=PCA-Part)": (_drp(delta=1.0), False),
        "A2: no shrinkage": (_drp(shrinkage="none"), False),
        "A2: fixed eps*I": (_drp(shrinkage="eps"), False),
        "A2: Ledoit-Wolf": (_drp(shrinkage="lw"), False),
        "A2: OAS": (_drp(shrinkage="oas"), False),
        "A3: PCA-Part + local Lloyd": (lambda X, K, s: B.init_pca_part(X, K, local_iters=5), True),
        "A4: random mode": (_drp(mode_policy="random"), False),
        "A5: Euclid in B/C": (_drp(metric="euclid"), False),
        "A6: full-space metric in C": (_drp(space="full"), False),
        "A7: rho-gate (rho=0.5)": (_drp(rho=0.5), False),
        "A8: SA (T=5, all axes)": (_drp(mode_policy="arbitrate"), True),
        "A8: SA (T=0, all axes)": (_drp(mode_policy="arbitrate", local_iters=0), True),
        "A8: SA (T=5, q<=3)": (_sa(), True),
        "A8: SA (T=5, q<=10)": (_sa(local_iters=5, max_axes=10), True),
        "A8: SA (T=0, q<=3)": (_sa(local_iters=0), True),
        "PCA-Part": (lambda X, K, s: B.init_pca_part(X, K), True),
    }


# ----------------------------------------------------------------------------
# Core evaluation
# ----------------------------------------------------------------------------

def run_one(fn, X, y, K, seed, with_sil=True):
    t0 = time.perf_counter()
    C = np.asarray(fn(X, K, seed), dtype=float)
    t_init = time.perf_counter() - t0
    init_sse = M.sse(X, C)
    km = KMeans(K, init=C, n_init=1, max_iter=MAX_ITER, tol=TOL, algorithm="lloyd",
                random_state=0).fit(X)
    ari, nmi = M.external_scores(y, km.labels_)
    return dict(init_sse=init_sse, sse=float(km.inertia_), n_iter=int(km.n_iter_), ari=ari,
                nmi=nmi, sil=M.silhouette(X, km.labels_) if with_sil else float('nan'),
                t_init=t_init)


def evaluate(name, X, y, K, methods, n_seeds, verbose=True, with_sil=True):
    rows = []
    for mname, (fn, det) in methods.items():
        cache = None
        for s in range(n_seeds):
            try:
                if det and cache is not None:
                    r = dict(cache)
                else:
                    r = run_one(fn, X, y, K, s, with_sil)
                    cache = r
            except Exception as e:  # keep the benchmark running
                warnings.warn(f"{mname} failed on {name} (seed {s}): {e}")
                break
            rows.append(dict(dataset=name, method=mname, seed=s, K=K, **r))
        if verbose and rows and rows[-1]["method"] == mname:
            sub = [r["sse"] for r in rows if r["method"] == mname]
            print(f"  {name:9s} {mname:28s} SSE={np.mean(sub):.5g}")
    return rows


def run_real(datasets, methods, n_seeds=30, verbose=True):
    rows = []
    for name, (X, y, K) in datasets.items():
        if verbose:
            print(f"[real] {name}  (n={len(X)}, d={X.shape[1]}, K={K})")
        rows += evaluate(name, X, y, K, methods, n_seeds, verbose)
    return add_normalised(pd.DataFrame(rows))


def add_normalised(df, group="dataset"):
    """nsse = SSE / best SSE on the dataset (over all methods and seeds);  fail = nsse > 1.5."""
    best = df.groupby(group)["sse"].transform("min")
    df = df.copy()
    df["nsse"] = df["sse"] / best
    df["fail"] = (df["nsse"] > 1.5).astype(float)       # paper's threshold (almost never triggers)
    df["fail2"] = (df["nsse"] > 1.02).astype(float)     # stricter: >2% above the best SSE
    return df


# ----------------------------------------------------------------------------
# Seed stability (bootstrap resamples)
# ----------------------------------------------------------------------------

def run_stability(datasets, methods, n_boot=15, max_n=4000, verbose=True):
    rows = []
    for name, (X, y, K) in datasets.items():
        rng = np.random.default_rng(0)
        if len(X) > max_n:
            X = X[rng.choice(len(X), max_n, replace=False)]
        diam = M.diameter(X)
        boots = [X[rng.choice(len(X), len(X), replace=True)] for _ in range(n_boot)]
        for mname, (fn, det) in methods.items():
            try:
                cs = [np.asarray(fn(Xb, K, 0), float) for Xb in boots]
                rows.append(dict(dataset=name, method=mname, stability=M.seed_stability(cs, diam)))
            except Exception as e:
                warnings.warn(f"stability {mname}/{name}: {e}")
        if verbose:
            print(f"[stability] {name} done")
    return pd.DataFrame(rows)


# ----------------------------------------------------------------------------
# Mode usage (Prediction P6)
# ----------------------------------------------------------------------------

def run_modes(datasets, seeds=(0, 1, 2), **drp_kw):
    rows = []
    for name, (X, y, K) in datasets.items():
        for s in seeds:
            m = DRPAP(random_state=s, **drp_kw)
            m.fit_init(X, K)
            c = m.mode_counts_
            tot = max(sum(c.values()), 1)
            rows.append(dict(dataset=name, seed=s, A=c["A"] / tot, B=c["B"] / tot,
                             C=c["C"] / tot, frac_BC=(c["B"] + c["C"]) / tot,
                             mean_rank=np.mean([t["r"] for t in m.trace_]) if m.trace_ else np.nan))
    return pd.DataFrame(rows).groupby("dataset").mean(numeric_only=True).drop(columns="seed").reset_index()


# ----------------------------------------------------------------------------
# Synthetic experiments (Predictions P1-P3)
# ----------------------------------------------------------------------------

SYN_METHODS = ["k-means++", "PCA-Part", "Var-Part", "k-means++ MD", "DRP-AP", "DRP-AP-SA",
               "DRP-AP-SA (T=0)"]
# second synthetic suite: generators that are NOT the Theorem-2 construction
SYN2_METHODS = ["k-means++", "Greedy k-means++", "PCA-Part", "PCA-Part (Su-Dy, mean split)",
                "PCA-Part + local Lloyd (A3)",
                "DRP-AP-SA", "DRP-AP-SA (T=0)", "Greedy k-means++ best-of-3", "SA pool-3"]


def synthetic2_grid(quick=False):
    g = []
    for r in (2, 3) if quick else (2, 3, 5):
        for decay in (1.0, 0.7, 0.4):
            g.append(("subspace_gmm", dict(r=r, decay=decay, K=8)))
    for overlap in (1.0, 2.0) if quick else (0.7, 1.0, 2.0):
        g.append(("random_gmm", dict(overlap=overlap)))
    g.append(("student", {}))
    for K in (6, 12):
        g.append(("ring", dict(K=K)))
    return g


def run_synthetic2(n_draws=30, quick=False, verbose=True, methods=None):
    """Same protocol as run_synthetic on generators unrelated to the Theorem-2 construction."""
    reg = main_methods()
    methods = {k: reg[k] for k in (methods or SYN2_METHODS)}
    rows = []
    for fam, params in synthetic2_grid(quick):
        cfg = ",".join(f"{k}={v:g}" if isinstance(v, (int, float)) else f"{k}={v}"
                       for k, v in params.items())
        for draw in range(n_draws):
            X, y, K = FAMILIES[fam](seed=draw, **params)
            rows += [dict(family=fam, config=cfg, draw=draw, **r)
                     for r in evaluate(f"{fam}[{cfg}]", X, y, K, methods, 1, verbose=False,
                                      with_sil=False)]
        if verbose:
            print(f"[synthetic2] {fam} {cfg} done")
    df = pd.DataFrame(rows)
    best = df.groupby(["family", "config", "draw"])["sse"].transform("min")
    df["nsse"] = df["sse"] / best
    return df


def synthetic_grid(quick=False):
    g = []
    g.append(("mono", dict(d=10, K=4)))
    for r in (2, 3) if quick else (2, 3, 5):
        for ratio in (0.2, 0.5, 0.8, 0.95):
            g.append(("multi_axial", dict(r=r, ratio=ratio)))
    for kappa in (1e2, 1e4) if quick else (1e2, 1e4, 1e6):
        g.append(("anisotropic", dict(kappa=kappa)))
    g.append(("shells", {}))
    for d in (20, 50) if quick else (20, 50, 100):
        for mult in (1, 2, 10):
            g.append(("highdim", dict(d=d, n=d * mult if mult > 1 else int(d * 1.2))))
    return g


def run_synthetic(n_draws=30, quick=False, verbose=True, methods=None):
    reg = main_methods()
    methods = {k: reg[k] for k in (methods or SYN_METHODS)}
    rows = []
    for fam, params in synthetic_grid(quick):
        cfg = ",".join(f"{k}={v:g}" if isinstance(v, (int, float)) else f"{k}={v}"
                       for k, v in params.items())
        for draw in range(n_draws):
            X, y, K = FAMILIES[fam](seed=draw, **params)
            rows += [dict(family=fam, config=cfg, draw=draw, **r)
                     for r in evaluate(f"{fam}[{cfg}]", X, y, K, methods, 1, verbose=False,
                                      with_sil=False)]
        if verbose:
            print(f"[synthetic] {fam} {cfg} done")
    df = pd.DataFrame(rows)
    best = df.groupby(["family", "config", "draw"])["sse"].transform("min")
    df["nsse"] = df["sse"] / best
    return df


# ----------------------------------------------------------------------------
# Ablations and sensitivity
# ----------------------------------------------------------------------------

def run_ablation(datasets, n_seeds=10, verbose=True):
    rows = []
    for name, (X, y, K) in datasets.items():
        if verbose:
            print(f"[ablation] {name}")
        rows += evaluate(name, X, y, K, ablation_methods(), n_seeds, verbose=False,
                         with_sil=False)
    df = pd.DataFrame(rows)
    ref = df[df.method == "DRP-AP (full)"].groupby("dataset")["sse"].mean()
    df["rel_sse"] = df["sse"] / df["dataset"].map(ref)
    return df


def run_sensitivity(datasets, n_seeds=5, deltas=(0.01, 0.05, 0.1, 0.2),
                    kappas=(1e2, 1e3, 1e4, 1e6), Ts=(0, 5, 10), verbose=True):
    rows = []
    for name, (X, y, K) in datasets.items():
        base = None
        for delta, kappa0, T in itertools.product(deltas, kappas, Ts):
            fn = _drp(delta=delta, kappa0=kappa0, local_iters=T)
            for s in range(n_seeds):
                r = run_one(fn, X, y, K, s, with_sil=False)
                m = DRPAP(delta=delta, kappa0=kappa0, local_iters=T, random_state=s)
                m.fit_init(X, K)
                c = m.mode_counts_
                rows.append(dict(dataset=name, delta=delta, kappa0=kappa0, T=T, seed=s,
                                 sse=r["sse"], frac_B=c["B"] / max(sum(c.values()), 1),
                                 frac_C=c["C"] / max(sum(c.values()), 1)))
        if verbose:
            print(f"[sensitivity] {name} done")
    df = pd.DataFrame(rows)
    df["nsse"] = df["sse"] / df.groupby("dataset")["sse"].transform("min")
    return df

In [ ]:
%%writefile drp_ap/reporting.py
"""Turn raw result tables into the tables / figures / statistics of Section 'Results'."""
from __future__ import annotations

import os

import numpy as np
import pandas as pd

from . import stats as S

ORDER_REAL = ["Iris", "Wine", "BC", "Digits", "MNIST", "F-MNIST", "COIL-20", "Olivetti",
              "20NG", "R8", "Isolet", "HAR"]
HEADER = {"Iris": "Iris", "Wine": "Wine", "BC": "BC", "Digits": "Digits", "MNIST": "MNIST",
          "F-MNIST": "F-MNIST", "COIL-20": "COIL", "Olivetti": "Olivetti", "20NG": "20NG",
          "R8": "R8", "Isolet": "Isolet", "HAR": "HAR"}


def _tex(s):
    return str(s).replace("_", r"\_").replace("&", r"\&").replace("+", r"$^{+}$") \
        if "+" in str(s) and "k-means" not in str(s) else str(s).replace("_", r"\_").replace("&", r"\&")


def sse_table(df, ref="DRP-AP", path=None):
    """Table 'tab:sse' of the paper: mean +- std of normalised SSE, bold best, dagger = significantly
    worse than `ref` (paired Wilcoxon over seeds, Holm across methods)."""
    from scipy import stats
    datasets = [d for d in ORDER_REAL if d in df.dataset.unique()] + \
               [d for d in df.dataset.unique() if d not in ORDER_REAL]
    methods = list(df.method.unique())
    mean = df.groupby(["method", "dataset"])["nsse"].mean().unstack()
    std = df.groupby(["method", "dataset"])["nsse"].std().unstack()
    sig = pd.DataFrame(False, index=methods, columns=datasets)
    for d in datasets:
        p = {}
        sub = df[df.dataset == d]
        a = sub[sub.method == ref].sort_values("seed")["nsse"].values
        for m in methods:
            if m == ref:
                continue
            b = sub[sub.method == m].sort_values("seed")["nsse"].values
            if len(a) == len(b) and len(a) > 5 and not np.allclose(a, b) and b.mean() > a.mean():
                try:
                    p[m] = stats.wilcoxon(b, a, alternative="greater").pvalue
                except ValueError:
                    p[m] = 1.0
        adj = S.holm(p) if p else {}
        for m, v in adj.items():
            sig.loc[m, d] = v < 0.05
    lines = [r"\begin{tabular}{l" + "c" * len(datasets) + "}", r"\toprule",
             "Method & " + " & ".join(HEADER.get(d, d) for d in datasets) + r" \\", r"\midrule"]
    for m in methods:
        cells = []
        for d in datasets:
            if d not in mean.columns or pd.isna(mean.loc[m, d]):
                cells.append("--")
                continue
            c = f"{mean.loc[m, d]:.3f}" + (f"$\\pm${std.loc[m, d]:.3f}" if not pd.isna(std.loc[m, d]) else "")
            if np.isclose(mean.loc[m, d], mean[d].min(), atol=5e-4):
                c = r"\textbf{" + c + "}"
            if sig.loc[m, d]:
                c += r"$^\dagger$"
            cells.append(c)
        lines.append(_tex(m) + " & " + " & ".join(cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}"]
    tex = "\n".join(lines)
    if path:
        open(path, "w").write(tex)
    return mean


def aggregate_table(df, path=None):
    g = df.groupby("method").agg(ARI=("ari", "mean"), NMI=("nmi", "mean"),
                                 Failure=("fail", "mean"), Fail2pct=("fail2", "mean"), Iterations=("n_iter", "mean"),
                                 Time_s=("t_init", "mean"), NSSE=("nsse", "mean"))
    if path:
        lines = [r"\begin{tabular}{lccccccc}", r"\toprule",
                 r"Method & NSSE & ARI & NMI & Fail$>$1.5 & Fail$>$1.02 & Iterations & Init time (s) \\", r"\midrule"]
        for m, r in g.iterrows():
            lines.append(f"{_tex(m)} & {r.NSSE:.3f} & {r.ARI:.3f} & {r.NMI:.3f} & {r.Failure:.3f} & {r.Fail2pct:.3f} "
                         f"& {r.Iterations:.1f} & {r.Time_s:.3f} " + r"\\")
        lines += [r"\bottomrule", r"\end{tabular}"]
        open(path, "w").write("\n".join(lines))
    return g


def statistical_summary(df, out_dir, ref="DRP-AP"):
    """Friedman + Nemenyi (critical-difference diagram) + Wilcoxon-Holm vs reference."""
    pivot = df.groupby(["dataset", "method"])["nsse"].mean().unstack().dropna(axis=1)
    res = {}
    if pivot.shape[0] >= 3 and pivot.shape[1] >= 3:
        stat, p = S.friedman(pivot)
        ranks = S.average_ranks(pivot)
        cd = S.nemenyi_cd(pivot.shape[1], pivot.shape[0])
        S.cd_diagram(ranks, cd, os.path.join(out_dir, "cd_diagram.png"),
                     f"Average ranks of normalised SSE (Friedman p={p:.3g})")
        ranks.to_csv(os.path.join(out_dir, "average_ranks.csv"), header=["avg_rank"])
        res.update(friedman_stat=stat, friedman_p=p, cd=cd, ranks=ranks)
    for r_ in dict.fromkeys([ref, "DRP-AP-SA", "Portfolio {PCA,Var,SA}"]):
        if r_ in pivot.columns:
            w = S.wilcoxon_vs_ref(pivot, r_)
            safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in r_)
            w.to_csv(os.path.join(out_dir, f"wilcoxon_holm_vs_{safe}.csv"))
            res["wilcoxon" if r_ == ref else f"wilcoxon::{r_}"] = w
    return res


def mode_correlation(modes_df, df, base="PCA-Part", ref="DRP-AP"):
    """P6: Spearman correlation between fraction of B/C splits and SSE gain over PCA-Part."""
    from scipy.stats import spearmanr
    m = df.groupby(["dataset", "method"])["nsse"].mean().unstack()
    if base not in m.columns or ref not in m.columns:
        return None          # needs both methods to have been run
    gain = (m[base] - m[ref]).rename("gain")
    j = modes_df.set_index("dataset").join(gain, how="inner").dropna()
    if len(j) < 4 or j["frac_BC"].nunique() < 2 or j["gain"].nunique() < 2:
        return None          # correlation undefined for constant input
    rho, p = spearmanr(j["frac_BC"], j["gain"])
    return dict(spearman_rho=float(rho), p=float(p), n=len(j))


def synthetic_report(df, out_dir):
    """Verdicts for P1-P3 and summary plots."""
    from scipy import stats
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    out = []
    piv = df.pivot_table(index=["family", "config", "draw"], columns="method", values="nsse")
    summ = df.groupby(["family", "config", "method"])["nsse"].agg(["mean", "std"]).unstack()
    summ.to_csv(os.path.join(out_dir, "synthetic_summary.csv"))

    def wil(cfg_mask, a, b, alt):
        sub = piv[cfg_mask]
        if len(sub) < 6:
            return float("nan")          # too few draws for a meaningful test
        if np.allclose(sub[a], sub[b]):
            return 1.0
        return float(stats.wilcoxon(sub[a], sub[b], alternative=alt).pvalue)

    fam = piv.index.get_level_values("family")
    cfg = piv.index.get_level_values("config")
    cands = [c for c in ("DRP-AP", "DRP-AP-SA", "DRP-AP-SA (T=0)") if c in piv.columns]
    na = lambda p: np.isnan(p)
    for cand in cands:
        # P1: mono-directional: the candidate must not be significantly worse than PCA-Part
        p1 = wil(fam == "mono", cand, "PCA-Part", "greater")
        out.append(f"[{cand}] P1 (mono): p(worse than PCA-Part)={p1:.3g} -> "
                   f"{'n/a (need >= 6 draws)' if na(p1) else ('REFUTED' if p1 < 0.05 else 'not refuted')}")
        # P2: multi-axial above the failure threshold of the construction
        # (a^2 > (2/pi) sigma1^2  <=>  ratio > 2/pi + sigma2^2 ~ 0.73 for sigma2=0.3):
        # the candidate must be significantly better than PCA-Part there.
        for c in sorted(set(cfg[fam == "multi_axial"])):
            ratio = float(c.split("ratio=")[1].split(",")[0]) if "ratio=" in c else 0
            p = wil(cfg == c, cand, "PCA-Part", "less")
            verdict = ("n/a (need >= 6 draws)" if na(p) else
                       "supported" if p < 0.05 else
                       ("NOT supported" if ratio > 0.75 else "n.s. (below threshold)"))
            out.append(f"[{cand}] P2 ({c}): p(better than PCA-Part)={p:.3g} -> {verdict}")
        # P3: SSE on anisotropic data vs K-means++ MD (seed stability is in stability.csv)
        for c in sorted(set(cfg[fam == "anisotropic"])):
            p = wil(cfg == c, cand, "k-means++ MD", "less")
            out.append(f"[{cand}] P3 SSE ({c}): p(better than k-means++ MD)={p:.3g}")
        # small-sample regime (n ~ d): is the candidate WORSE than PCA-Part?
        for c in sorted(set(cfg[fam == "highdim"])):
            p = wil(cfg == c, cand, "PCA-Part", "greater")
            out.append(f"[{cand}] small-sample ({c}): p(worse than PCA-Part)={p:.3g}"
                       + ("  <- significantly worse" if (not na(p) and p < 0.05) else ""))
    open(os.path.join(out_dir, "synthetic_verdicts.txt"), "w").write("\n".join(out))

    # plot: nsse vs lambda2/lambda1 for multi-axial
    sub = df[df.family == "multi_axial"].copy()
    if len(sub):
        sub["r"] = sub.config.str.extract(r"r=(\d+)").astype(int)
        sub["ratio"] = sub.config.str.extract(r"ratio=([\d.]+)").astype(float)
        rs = sorted(sub.r.unique())
        fig, axs = plt.subplots(1, len(rs), figsize=(4.2 * len(rs), 3.4), sharey=True)
        axs = np.atleast_1d(axs)
        for ax, r in zip(axs, rs):
            for m, g in sub[sub.r == r].groupby("method"):
                a = g.groupby("ratio")["nsse"].mean()
                ax.plot(a.index, a.values, marker="o", label=m, lw=2.2 if m.startswith("DRP-AP") else 1)
            ax.set_title(f"r = {r}  (K = {2**r})")
            ax.set_xlabel(r"$\lambda_2/\lambda_1$")
        axs[0].set_ylabel("normalised SSE")
        axs[-1].legend(fontsize=7)
        fig.tight_layout()
        fig.savefig(os.path.join(out_dir, "synthetic_multiaxial.png"), dpi=150)
        plt.close(fig)
    return out


def save_all(out_dir, **tables):
    os.makedirs(out_dir, exist_ok=True)
    for k, v in tables.items():
        if isinstance(v, pd.DataFrame):
            v.to_csv(os.path.join(out_dir, f"{k}.csv"), index=False)


def synthetic2_report(df, out_dir):
    """Non-circular synthetic suite: per config, mean NSSE of each method, and Wilcoxon p-values of
    DRP-AP-SA (T=5) and the SA pool against PCA-Part and against Greedy k-means++ best-of-3."""
    from scipy import stats
    piv = df.pivot_table(index=["family", "config", "draw"], columns="method", values="nsse")
    summ = df.groupby(["family", "config", "method"])["nsse"].mean().unstack()
    summ.to_csv(os.path.join(out_dir, "synthetic2_summary.csv"))
    out = []
    cfgs = sorted(set(piv.index.get_level_values("config")))
    fam = piv.index.get_level_values("family")
    cfg = piv.index.get_level_values("config")
    pairs = [("DRP-AP-SA", "PCA-Part"), ("DRP-AP-SA", "PCA-Part (Su-Dy, mean split)"), ("DRP-AP-SA", "PCA-Part + local Lloyd (A3)"),
             ("DRP-AP-SA", "Greedy k-means++"), ("SA pool-3", "Greedy k-means++ best-of-3")]
    rows = []
    for c in cfgs:
        sub = piv[cfg == c]
        for a, b in pairs:
            if a not in sub or b not in sub or len(sub) < 6:
                continue
            d = (sub[a] - sub[b]).to_numpy()
            if np.allclose(d, 0):
                p_less = p_gr = 1.0
            else:
                p_less = float(stats.wilcoxon(sub[a], sub[b], alternative="less").pvalue)
                p_gr = float(stats.wilcoxon(sub[a], sub[b], alternative="greater").pvalue)
            rows.append(dict(config=c, a=a, b=b, mean_diff=float(d.mean()), p_a_better=p_less,
                             p_a_worse=p_gr))
            out.append(f"[{c}] {a} vs {b}: mean dNSSE={d.mean():+.4f}  "
                       f"p(a better)={p_less:.3g}  p(a worse)={p_gr:.3g}")
    pd.DataFrame(rows).to_csv(os.path.join(out_dir, "synthetic2_tests.csv"), index=False)
    return out

In [ ]:
%%writefile drp_ap/tuning.py
"""Held-out choice of the DRP-AP-SA hyper-parameters (T, q_max) by leave-one-dataset-out (LODO).

For every dataset d, the configuration with the lowest mean normalised SSE over the OTHER datasets
is selected and evaluated on d. The held-out score therefore never uses d for choosing (T, q_max),
which removes the objection that the default was tuned on the evaluation benchmark.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

from . import baselines as B
from .core import DRPAPSA
from .experiments import evaluate

DEFAULT_TS = (0, 2, 5, 10)
DEFAULT_QS = (1, 2, 3, 5)


def cfg_name(T, q):
    return f"SA[T={T},q={q}]"


def grid_methods(Ts=DEFAULT_TS, qs=DEFAULT_QS):
    m = {cfg_name(T, q): ((lambda X, K, s, T=T, q=q:
                           DRPAPSA(local_iters=T, max_axes=q).fit_init(X, K)), True)
         for T in Ts for q in qs}
    m["PCA-Part"] = (lambda X, K, s: B.init_pca_part(X, K), True)
    return m


def run_grid(datasets, Ts=DEFAULT_TS, qs=DEFAULT_QS, verbose=False):
    """Deterministic configurations: one run each. Returns long df (dataset, method, sse)."""
    rows = []
    for name, (X, y, K) in datasets.items():
        rows += evaluate(name, X, y, K, grid_methods(Ts, qs), 1, verbose=verbose, with_sil=False)
    df = pd.DataFrame(rows)
    df["nsse"] = df["sse"] / df.groupby("dataset")["sse"].transform("min")
    return df


def lodo(df, default=cfg_name(5, 3), pool=None, ref="PCA-Part"):
    """df: columns dataset, method, nsse (e.g. from run_grid or from per-method means).
    Returns (per-dataset table, summary dict)."""
    piv = df.groupby(["dataset", "method"])["nsse"].mean().unstack()
    pool = pool or [c for c in piv.columns if c != ref]
    pool = [c for c in pool if c in piv.columns]
    rows = []
    for d in piv.index:
        others = piv.drop(index=d)
        # re-normalise on the other datasets only (their own best), then rank by mean
        pick = others[pool].mean().idxmin()
        rows.append(dict(dataset=d, selected=pick, lodo=piv.loc[d, pick],
                         fixed_default=piv.loc[d, default] if default in piv else np.nan,
                         reference=piv.loc[d, ref] if ref in piv else np.nan,
                         oracle=piv.loc[d, pool].min()))
    t = pd.DataFrame(rows).set_index("dataset")
    summ = {c: float(t[c].mean()) for c in ("lodo", "fixed_default", "reference", "oracle")}
    summ["global_best_in_sample"] = piv[pool].mean().idxmin()
    for c in ("fixed_default", "reference"):
        diff = t["lodo"] - t[c]
        if np.any(diff != 0) and len(diff) >= 6:
            summ[f"wilcoxon_p_lodo_vs_{c}"] = float(wilcoxon(diff).pvalue)
    return t, summ

In [ ]:
%%writefile drp_ap/diagnostics.py
"""Measured directional rank on real datasets and diagnostics of where DRP-AP-SA helps or hurts.

For every dataset: spectrum of the root cluster, mean r_delta and lambda2/lambda1 along the splits
actually made, fraction of splits whose chosen axis is not the first principal axis, and the
initial and final SSE of PCA-Part and DRP-AP-SA after one Lloyd run.  `correlations` relates these
to the final-SSE gain of DRP-AP-SA over PCA-Part (exploratory with few datasets: report effect sizes,
not p-values)."""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from . import baselines as B
from . import metrics as M
from .core import DRPAPSA, directional_rank, spectrum
from .experiments import _final_sse


def diagnose(X, K, delta=0.05):
    Xc = X - X.mean(0)
    lam, _ = spectrum(Xc)
    m = DRPAPSA()
    C_sa = m.fit_init(X, K)
    axes = [t["axis"] for t in m.trace_]
    C_pca = B.init_pca_part(X, K)
    f_sa, f_pca = _final_sse(X, K, C_sa), _final_sse(X, K, C_pca)
    i_sa, i_pca = M.sse(X, C_sa), M.sse(X, C_pca)
    return dict(n=len(X), d=X.shape[1], K=K,
                root_r=directional_rank(lam, delta),
                root_ratio=float(lam[1] / lam[0]) if len(lam) > 1 else 0.0,
                mean_r=float(np.mean([t["r"] for t in m.trace_])) if m.trace_ else np.nan,
                mean_ratio=float(np.mean([t["ratio"] for t in m.trace_])) if m.trace_ else np.nan,
                frac_not_u1=float(np.mean([a != 0 for a in axes])) if axes else 0.0,
                init_ratio=i_sa / i_pca, final_ratio=f_sa / f_pca,
                gain_pct=100.0 * (1.0 - f_sa / f_pca))


def run_diagnostics(datasets, delta=0.05):
    rows = [dict(dataset=n, **diagnose(X, K, delta)) for n, (X, y, K) in datasets.items()]
    return pd.DataFrame(rows)


def correlations(df):
    """Spearman correlations (rho, p) of the gain with the measured quantities."""
    out = {}
    if len(df) < 4:
        return out
    for c in ("root_ratio", "root_r", "mean_r", "mean_ratio", "frac_not_u1", "K", "n", "d"):
        if df[c].nunique() > 1:
            rho, p = spearmanr(df[c], df["gain_pct"])
            out[c] = (float(rho), float(p))
    # does the initial-SSE improvement predict the final-SSE improvement?
    if df["init_ratio"].nunique() > 1:
        rho, p = spearmanr(1 - df["init_ratio"], df["gain_pct"])
        out["init_gain_vs_final_gain"] = (float(rho), float(p))
    return out

In [ ]:
%%writefile run_experiments.py
#!/usr/bin/env python
"""Run the full experimental protocol of the DRP-AP paper.

Examples
--------
  python run_experiments.py --quick                      # fast smoke run (offline datasets only)
  python run_experiments.py --datasets all --seeds 30    # full protocol on 12 datasets
  python run_experiments.py --datasets Iris Wine Digits --seeds 10 --synthetic --ablation
"""
import argparse
import json
import os
import time
import warnings

import numpy as np
import pandas as pd

_REQUIRED = ["__init__", "core", "baselines", "metrics", "stats", "synthetic", "data",
             "experiments", "reporting", "tuning", "diagnostics"]
_here = os.path.join(os.path.dirname(os.path.abspath(__file__)), "drp_ap")
_missing = [f"drp_ap/{m}.py" for m in _REQUIRED if not os.path.exists(os.path.join(_here, m + ".py"))]
if _missing:
    raise SystemExit("Missing package files: " + ", ".join(_missing) +
                     "\nIn Colab, run every '%%writefile' cell (Runtime > Run all) so that they are "
                     "written to /content/drp_ap/ before launching this script.")

from drp_ap import experiments as E
from drp_ap import reporting as R
from drp_ap.data import ALL_NAMES, DATASETS, DEFAULT_DIR, load_many, manifest

OFFLINE = ["Iris", "Wine", "BC", "Digits"]


def _resumable(path, fn, data, **kw):
    """Run fn(one dataset, **kw) dataset by dataset, appending to `path` after each one and skipping
    the datasets already in the file, so that a crashed / timed-out Colab session can be resumed."""
    done = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    have = set(done["dataset"]) if len(done) else set()
    for name, d in data.items():
        if name in have:
            print(f"[resume] {name}: already in {os.path.basename(path)}")
            continue
        part = fn({name: d}, **kw)
        done = pd.concat([done, part], ignore_index=True)
        done.to_csv(path, index=False)            # checkpoint after every dataset
    return done


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--datasets", nargs="+", default=["all"],
                    help="names from %s, 'all' (12 core), 'extended'/'benchmark30' (30) or 'offline'" % ALL_NAMES)
    ap.add_argument("--methods", nargs="+", default=None, help="subset of method names")
    ap.add_argument("--seeds", type=int, default=30)
    ap.add_argument("--max-n", type=int, default=20000,
                    help="stratified subsample cap per dataset (0 = no cap)")
    ap.add_argument("--data-dir", default=DEFAULT_DIR)
    ap.add_argument("--out", default="results")
    ap.add_argument("--quick", action="store_true", help="few seeds / small grids")
    ap.add_argument("--synthetic", action="store_true")
    ap.add_argument("--ablation", action="store_true")
    ap.add_argument("--sensitivity", action="store_true")
    ap.add_argument("--stability", action="store_true")
    ap.add_argument("--no-restarts", action="store_true",
                    help="skip the equal-compute 'Greedy k-means++ best-of-3/10' baselines (they add "
                         "3 + 10 full Lloyd runs per seed and dominate the run time)")
    ap.add_argument("--synthetic2", action="store_true",
                    help="second synthetic suite (generators unrelated to the Theorem-2 construction)")
    ap.add_argument("--tune", action="store_true",
                    help="leave-one-dataset-out choice of (T, q_max) on a deterministic grid")
    ap.add_argument("--diagnostics", action="store_true",
                    help="measured directional rank per dataset and correlation with the SA gain")
    ap.add_argument("--skip-main", action="store_true",
                    help="do not re-run the main real benchmark / mode usage (use it to finish or "
                         "resume the extras: --skip-main --ablation --sensitivity)")
    ap.add_argument("--all-extras", action="store_true",
                    help="synthetic + synthetic2 + ablation + sensitivity + stability + tune + diagnostics")
    a = ap.parse_args()

    if a.all_extras:
        a.synthetic = a.ablation = a.sensitivity = a.stability = True
        a.synthetic2 = a.tune = a.diagnostics = True
    if a.quick:
        a.seeds = min(a.seeds, 3)
        if a.datasets == ["all"]:
            a.datasets = ["offline"]
    names = (list(DATASETS) if a.datasets == ["all"] else
             ALL_NAMES if a.datasets in (["extended"], ["benchmark30"]) else
             OFFLINE if a.datasets == ["offline"] else a.datasets)
    os.makedirs(a.out, exist_ok=True)
    t0 = time.time()
    data = load_many(names, a.data_dir, a.max_n or None)
    if not data:
        raise SystemExit("No dataset could be loaded.")
    man = manifest(data)
    man.to_csv(os.path.join(a.out, "datasets_manifest.csv"), index=False)
    miss = [n for n in names if n not in data]
    print(f"\n=== Loaded {len(data)}/{len(names)} datasets ===\n", man.to_string(index=False))
    if miss:
        print("NOT LOADED:", miss, "-> add <data-dir>/<name>.npz (arrays X, y) and re-run to include them")

    reg = E.main_methods()
    methods = reg if a.methods is None else {k: reg[k] for k in a.methods}
    if a.no_restarts:
        methods = {k: v for k, v in methods.items() if "best-of" not in k}

    df = None
    if not a.skip_main:
        # ---------------- real benchmarks (Tables sse / agg, statistics) -------------
        df = E.run_real(data, methods, a.seeds)
        df.to_csv(os.path.join(a.out, "real_raw.csv"), index=False)
        R.sse_table(df, path=os.path.join(a.out, "tab_sse.tex"))
        agg = R.aggregate_table(df, path=os.path.join(a.out, "tab_agg.tex"))
        agg.to_csv(os.path.join(a.out, "aggregate.csv"))
        stat = R.statistical_summary(df, a.out)
        print("\n=== Aggregate (mean over datasets and seeds) ===\n", agg.round(4).to_string())
        if "friedman_p" in stat:
            print(f"\nFriedman chi2={stat['friedman_stat']:.2f}  p={stat['friedman_p']:.3g}  "
                  f"Nemenyi CD={stat['cd']:.2f}\n", stat["ranks"].round(2).to_string())
        for k_, v_ in stat.items():
            if k_ == "wilcoxon" or k_.startswith("wilcoxon::"):
                who = "DRP-AP" if k_ == "wilcoxon" else k_.split("::", 1)[1]
                print(f"\nWilcoxon-Holm vs {who} (normalised SSE across datasets; diff>0 => other worse)\n",
                      v_.round(4).to_string())

        # ---------------- mode usage + P6 -------------------------------------------
        modes = E.run_modes(data)
        modes.to_csv(os.path.join(a.out, "mode_usage.csv"), index=False)
        print("\n=== Mode usage (fraction of splits) ===\n", modes.round(3).to_string(index=False))
        corr = R.mode_correlation(modes, df)
        if corr:
            print("P6 Spearman(frac B/C, gain over PCA-Part):", corr)
            json.dump(corr, open(os.path.join(a.out, "p6_correlation.json"), "w"))

    # ---------------- optional extras ---------------------------------------------
    if a.stability:
        stab_methods = {k: reg[k] for k in ["k-means++", "PCA-Part", "Var-Part", "k-means++ MD",
                                            "k-means++ MD Hybrid", "DRP-AP", "DRP-AP-SA", "DRP-AP-SA (T=0)"]}
        st = E.run_stability(data, stab_methods, n_boot=8 if a.quick else 15)
        st.to_csv(os.path.join(a.out, "stability.csv"), index=False)
        print("\n=== Seed stability (lower = better) ===\n",
              st.pivot(index="dataset", columns="method", values="stability").round(4).to_string())
    if a.synthetic:
        sdf = E.run_synthetic(n_draws=5 if a.quick else 30, quick=a.quick)
        sdf.to_csv(os.path.join(a.out, "synthetic_raw.csv"), index=False)
        print("\n=== Synthetic verdicts ===")
        print("\n".join(R.synthetic_report(sdf, a.out)))
    if a.synthetic2:
        s2 = E.run_synthetic2(n_draws=5 if a.quick else 30, quick=a.quick)
        s2.to_csv(os.path.join(a.out, "synthetic2_raw.csv"), index=False)
        print("\n=== Synthetic suite 2 (non-Theorem-2 generators) ===")
        print("\n".join(R.synthetic2_report(s2, a.out)))
    if a.tune:
        from drp_ap import tuning as TU
        gr = TU.run_grid(data)
        gr.to_csv(os.path.join(a.out, "tuning_grid_raw.csv"), index=False)
        lt, ls = TU.lodo(gr)
        lt.to_csv(os.path.join(a.out, "lodo_table.csv"))
        json.dump(ls, open(os.path.join(a.out, "lodo_summary.json"), "w"), indent=1)
        print("\n=== Leave-one-dataset-out choice of (T, q_max) ===\n", lt.round(4).to_string(),
              "\n", ls)
    if a.diagnostics:
        from drp_ap import diagnostics as DG
        dg = DG.run_diagnostics(data)
        dg.to_csv(os.path.join(a.out, "diagnostics.csv"), index=False)
        print("\n=== Measured directional rank and SA gain over PCA-Part ===\n", dg.round(3).to_string(index=False))
        print("Spearman (rho, p):", DG.correlations(dg))
        json.dump(DG.correlations(dg), open(os.path.join(a.out, "diagnostics_corr.json"), "w"))
    if a.ablation:
        ab = _resumable(os.path.join(a.out, "ablation_raw.csv"), E.run_ablation, data,
                        n_seeds=3 if a.quick else 10)
        tab = ab.groupby(["dataset", "method"])["rel_sse"].mean().unstack()
        tab.to_csv(os.path.join(a.out, "ablation_table.csv"))
        print("\n=== Ablations (SSE relative to full DRP-AP; <1 is better) ===\n",
              tab.round(4).to_string())
    if a.sensitivity:
        se = _resumable(os.path.join(a.out, "sensitivity_raw.csv"), E.run_sensitivity, data,
                        n_seeds=2 if a.quick else 5,
                        deltas=(0.05, 0.2) if a.quick else (0.01, 0.05, 0.1, 0.2),
                        kappas=(1e3, 1e6) if a.quick else (1e2, 1e3, 1e4, 1e6),
                        Ts=(0, 5) if a.quick else (0, 5, 10))
        print("\n=== Sensitivity: mean normalised SSE by delta / kappa0 / T ===")
        for k in ("delta", "kappa0", "T"):
            print(se.groupby(k)["nsse"].mean().round(4).to_string(), "\n")

    print(f"Done in {time.time() - t0:.0f}s. Results in ./{a.out}/")


if __name__ == "__main__":
    warnings.filterwarnings("ignore", category=FutureWarning)
    main()

In [ ]:
%%writefile tools/merge_results.py
"""Merge several run_experiments outputs (e.g. one Colab session per group of datasets) into ONE
benchmark: concatenates real_raw.csv, rebuilds the tables, the Friedman/Nemenyi/Wilcoxon-Holm
statistics, the leave-one-dataset-out analysis, and the dataset manifest.

    python tools/merge_results.py --out results_all results_part1 results_part2 results_part3
"""
import argparse
import json
import os
import sys

import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from drp_ap import reporting as R          # noqa: E402
from drp_ap import tuning as TU            # noqa: E402


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("dirs", nargs="+")
    ap.add_argument("--out", default="results_all")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    parts = [pd.read_csv(os.path.join(d, "real_raw.csv")) for d in a.dirs]
    df = pd.concat(parts, ignore_index=True)
    dup = df.duplicated(["dataset", "method", "seed"]).sum()
    if dup:
        print(f"warning: {dup} duplicated (dataset, method, seed) rows -> keeping the last")
        df = df.drop_duplicates(["dataset", "method", "seed"], keep="last")
    # a method missing on some datasets would make Friedman drop it: report it
    cnt = df.groupby("method")["dataset"].nunique()
    full = cnt.max()
    if (cnt < full).any():
        print("methods not run on every dataset (dropped from the rank statistics):",
              cnt[cnt < full].to_dict())
    df.to_csv(os.path.join(a.out, "real_raw.csv"), index=False)
    for d in a.dirs:
        m = os.path.join(d, "datasets_manifest.csv")
        if os.path.exists(m):
            pd.read_csv(m).to_csv(os.path.join(a.out, f"datasets_manifest_{os.path.basename(d.rstrip('/'))}.csv"),
                                  index=False)
    R.sse_table(df, path=os.path.join(a.out, "tab_sse.tex"))
    agg = R.aggregate_table(df, path=os.path.join(a.out, "tab_agg.tex"))
    agg.to_csv(os.path.join(a.out, "aggregate.csv"))
    stat = R.statistical_summary(df, a.out)
    print(f"{df.dataset.nunique()} datasets, {df.method.nunique()} methods, {df.seed.nunique()} seeds")
    print(agg.round(4).to_string())
    if "friedman_p" in stat:
        print(f"\nFriedman chi2={stat['friedman_stat']:.2f} p={stat['friedman_p']:.3g} CD={stat['cd']:.2f}")
        print(stat["ranks"].round(2).to_string())
    pool = [c for c in ("DRP-AP-SA", "DRP-AP-SA (T=0)", "DRP-AP-SA (mean split)") if c in set(df.method)]
    if len(pool) >= 2 and df.dataset.nunique() >= 6:
        t, s = TU.lodo(df, default="DRP-AP-SA", pool=pool, ref="PCA-Part")
        t.to_csv(os.path.join(a.out, "lodo_table.csv"))
        json.dump(s, open(os.path.join(a.out, "lodo_summary.json"), "w"), indent=1)
        print("\nLODO:", s)
    print("written to", a.out)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile tests/test_drp_ap.py
import numpy as np
import pytest

from drp_ap import DRPAP, directional_rank, conditioning_index
from drp_ap import baselines as B
from drp_ap import synthetic as S
from drp_ap.core import shrinkage_alpha, spectrum
from drp_ap.metrics import matched_distance


def test_directional_rank_definition():
    lam = np.array([0.7, 0.2, 0.05, 0.05])
    assert directional_rank(lam, 0.05) == 3      # 0.95 reached at r=3
    assert directional_rank(lam, 0.5) == 1
    assert directional_rank(lam, 1.0) == 1       # delta=1 -> always rank one
    assert directional_rank(lam, 1e-9) == 4


def test_conditioning_infinite_when_rank_deficient():
    assert conditioning_index(np.array([2.0, 1.0]), m=5, d=10) == float("inf")
    assert conditioning_index(np.array([4.0, 1.0]), m=50, d=2) == pytest.approx(4.0)


def test_proposition_containment_delta_one_equals_pca_part():
    X, y, K = S.multi_axial(n=600, r=2, ratio=0.6, seed=1)
    c1 = DRPAP(delta=1.0, random_state=0).fit_init(X, K)
    c2 = B.init_pca_part(X, K)
    np.testing.assert_allclose(np.sort(c1, axis=0), np.sort(c2, axis=0), atol=1e-9)


def test_mode_a_only_policy_equals_pca_part():
    X, y, K = S.anisotropic(n=500, d=8, K=4, seed=2)
    c1 = DRPAP(mode_policy="A_only", random_state=0).fit_init(X, K)
    np.testing.assert_allclose(np.sort(c1, axis=0), np.sort(B.init_pca_part(X, K), axis=0), atol=1e-9)


@pytest.mark.parametrize("kw", [{}, {"adaptive_k": True}, {"space": "full"},
                                {"shrinkage": "oas"}, {"shrinkage": "eps"}, {"metric": "euclid"},
                                {"mode_policy": "random"}])
def test_shapes_and_finiteness(kw):
    X, y, K = S.high_dim_mixture(n=120, d=60, K=5, seed=3)   # n ~ d: small-sample regime
    C = DRPAP(random_state=0, **kw).fit_init(X, K)
    assert C.shape == (K, 60) and np.isfinite(C).all()


def test_modes_used_in_deficiency_region():
    X, y, K = S.multi_axial(n=2000, d=20, r=2, ratio=0.8, seed=0)
    m = DRPAP(random_state=0)
    m.fit_init(X, K)
    counts = m.mode_counts_
    assert counts["B"] + counts["C"] >= 1          # multi-axial -> Modes B/C triggered


def test_mono_directional_is_mode_a_dominated():
    X, y, K = S.mono_directional(n=1200, d=6, K=4, seed=0)
    m = DRPAP(delta=0.2, random_state=0)
    m.fit_init(X, K)
    assert m.mode_counts_["A"] >= 1


def test_multi_axial_spectrum_matches_construction():
    r, ratio = 2, 0.6
    X, y, K = S.multi_axial(n=40000, d=10, r=r, ratio=ratio, seed=0)
    lam, _ = spectrum(X - X.mean(0))
    assert lam[1] / lam[0] == pytest.approx(ratio, rel=0.08)
    assert K == 4


def test_theorem2_construction_pca_part_worse_than_u2_split():
    """Corrected Theorem 2: PCA-Part is suboptimal iff a^2 > (2/pi) sigma1^2 while lam2 < lam1,
    i.e. only once lam2/lam1 exceeds ~2/pi (here ratio=0.9, sigma2=0.2 -> a^2=0.86 > 0.637)."""
    X, y, K = S.multi_axial(n=4000, d=2 + 1, r=1, ratio=0.9, sigma1=1.0, sigma2=0.2, noise=0.01, seed=0)
    Xc = X - X.mean(0)
    lam, Vt = spectrum(Xc)

    def split_sse(u):
        p = Xc @ u
        L, R = Xc[p <= np.median(p)], Xc[p > np.median(p)]
        return sum(((A - A.mean(0)) ** 2).sum() for A in (L, R))
    assert split_sse(Vt[1]) < split_sse(Vt[0])


def test_theorem2_below_threshold_pca_part_is_fine():
    """Below ratio ~ 2/pi the construction gives NO failure of PCA-Part (documents the regime)."""
    X, y, K = S.multi_axial(n=4000, d=3, r=1, ratio=0.5, sigma1=1.0, sigma2=0.3, noise=0.01, seed=0)
    Xc = X - X.mean(0)
    lam, Vt = spectrum(Xc)

    def split_sse(u):
        p = Xc @ u
        L, R = Xc[p <= np.median(p)], Xc[p > np.median(p)]
        return sum(((A - A.mean(0)) ** 2).sum() for A in (L, R))
    assert split_sse(Vt[0]) <= split_sse(Vt[1])


def test_shrinkage_alpha_in_unit_interval():
    X = np.random.default_rng(0).standard_normal((30, 40))
    a = shrinkage_alpha(X - X.mean(0), "lw")
    assert 0.0 <= a <= 1.0


def test_baselines_shapes():
    X, y, K = S.anisotropic(n=300, d=6, K=3, kappa=1e3, seed=0)
    rng = np.random.default_rng(0)
    for f in (B.init_random, B.init_kmeanspp, B.init_greedy_kmeanspp, B.init_bradley_fayyad,
              B.init_kmeanspp_md, B.init_kmeanspp_md_best, B.init_kmeanspp_md_hybrid, B.init_gmm):
        C = f(X, K, rng)
        assert C.shape == (K, 6), f.__name__
    for f in (B.init_kkz, B.init_pca_part, B.init_var_part):
        assert f(X, K).shape == (K, 6), f.__name__


def test_matched_distance_permutation_invariant():
    A = np.random.default_rng(0).standard_normal((5, 3))
    assert matched_distance(A, A[::-1]) == pytest.approx(0.0, abs=1e-12)


def test_arbitrate_first_split_dominates_pca_part_and_is_deterministic():
    from drp_ap.core import DRPAP as D
    X, y, K = S.multi_axial(n=1500, d=20, r=2, ratio=0.95, seed=3)
    c1 = D(mode_policy="arbitrate", local_iters=0).fit_init(X, 2)
    c2 = B.init_pca_part(X, 2)
    def sse2(C):
        d = ((X[:, None, :] - C[None]) ** 2).sum(2)
        return d.min(1).sum()
    assert sse2(c1) <= sse2(c2) + 1e-9          # K=2: best-axis split never worse than u1 split
    a = D(mode_policy="arbitrate").fit_init(X, 5)
    b = D(mode_policy="arbitrate").fit_init(X, 5)
    np.testing.assert_array_equal(a, b)


def test_max_axes_cap_is_valid_and_matches_when_not_binding():
    X, y, K = S.multi_axial(n=1200, d=20, r=2, ratio=0.9, seed=1)
    full = DRPAP(mode_policy="arbitrate", random_state=0).fit_init(X, K)
    cap = DRPAP(mode_policy="arbitrate", max_axes=50, random_state=0).fit_init(X, K)   # cap >= r
    np.testing.assert_allclose(full, cap)
    c1 = DRPAP(mode_policy="arbitrate", max_axes=1, random_state=0).fit_init(X, K)    # = PCA-Part axis
    assert c1.shape == (K, 20) and np.isfinite(c1).all()


def test_portfolio_never_worse_than_its_candidates_after_lloyd():
    from sklearn.cluster import KMeans
    from drp_ap.experiments import init_portfolio
    X, y, K = S.multi_axial(n=1500, d=15, r=2, ratio=0.95, seed=4)

    def final(C):
        return KMeans(K, init=C, n_init=1, algorithm="lloyd", random_state=0).fit(X).inertia_
    port = final(init_portfolio(X, K))
    assert port <= final(B.init_pca_part(X, K)) + 1e-6
    assert port <= final(B.init_var_part(X, K)) + 1e-6
    assert port <= final(DRPAP(mode_policy="arbitrate").fit_init(X, K)) + 1e-6


def test_drpapsa_defaults_and_equivalences():
    from drp_ap import DRPAPSA
    X, y, K = S.multi_axial(n=1200, d=20, r=2, ratio=0.9, seed=2)
    m = DRPAPSA()
    assert m.local_iters == 5 and m.max_axes == 3 and m.mode_policy == "arbitrate"
    a = m.fit_init(X, K)
    b = DRPAP(mode_policy="arbitrate", local_iters=5, max_axes=3).fit_init(X, K)
    np.testing.assert_allclose(a, b)                       # same as the explicit configuration
    np.testing.assert_array_equal(a, DRPAPSA().fit_init(X, K))   # deterministic
    assert DRPAPSA(mode_policy="random").mode_policy == "arbitrate"   # cannot be overridden


def test_best_of_restarts_never_worse_than_single_run_and_is_seeded():
    from sklearn.cluster import KMeans
    from drp_ap.experiments import best_of
    X, y, K = S.anisotropic(n=500, d=8, K=5, kappa=1e2, seed=1)

    def final(C):
        return KMeans(K, init=C, n_init=1, algorithm="lloyd", random_state=0).fit(X).inertia_
    f1 = best_of(B.init_greedy_kmeanspp, 1)
    f10 = best_of(B.init_greedy_kmeanspp, 10)
    assert final(f10(X, K, 0)) <= final(f1(X, K, 0)) + 1e-9          # first draw is shared (same seed)
    np.testing.assert_array_equal(f10(X, K, 3), f10(X, K, 3))
    assert f10(X, K, 0).shape == (K, 8)


def test_registry_contains_new_baselines():
    from drp_ap.experiments import main_methods, ablation_methods
    mm = main_methods()
    for k in ["PCA-Part (Su-Dy, mean split)", "Var-Part (Su-Dy, mean split)", "DRP-AP-SA (mean split)",
              "DRP-AP-SA", "DRP-AP-SA (T=0)", "PCA-Part + local Lloyd (A3)",
              "Greedy k-means++ best-of-3", "Greedy k-means++ best-of-10", "Portfolio {PCA,Var,SA}"]:
        assert k in mm, k
    assert "A8: SA (T=0, q<=3)" in ablation_methods()


def test_randomised_sa_is_seeded_diverse_and_valid():
    from drp_ap import DRPAPSAR
    X, y, K = S.multi_axial(n=1200, d=20, r=3, ratio=0.95, seed=1)
    a = DRPAPSAR(random_state=1).fit_init(X, K)
    np.testing.assert_array_equal(a, DRPAPSAR(random_state=1).fit_init(X, K))   # seeded
    assert not np.allclose(a, DRPAPSAR(random_state=2).fit_init(X, K))          # diverse
    assert a.shape == (K, 20) and np.isfinite(a).all()


def test_pool_never_worse_than_its_deterministic_anchor():
    from drp_ap import DRPAPSA
    from drp_ap.experiments import best_of_pool, _sar, _final_sse
    X, y, K = S.multi_axial(n=900, d=15, r=2, ratio=0.9, seed=5)
    anchor = lambda X, K: DRPAPSA().fit_init(X, K)
    pool = best_of_pool(anchor, _sar, 4)
    assert _final_sse(X, K, pool(X, K, 0)) <= _final_sse(X, K, anchor(X, K)) + 1e-9


@pytest.mark.parametrize("fam", ["subspace_gmm", "random_gmm", "student", "ring"])
def test_new_generators_shapes(fam):
    X, y, K = S.FAMILIES[fam](seed=0)
    assert X.shape[0] == len(y) and np.isfinite(X).all() and K == len(np.unique(y))


def test_lodo_selects_without_using_heldout_dataset():
    import pandas as pd
    from drp_ap.tuning import lodo
    rows = []
    for d, (a, b) in {"d1": (1.0, 1.1), "d2": (1.0, 1.1), "d3": (1.5, 1.0)}.items():
        rows += [dict(dataset=d, method="A", nsse=a), dict(dataset=d, method="B", nsse=b),
                 dict(dataset=d, method="PCA-Part", nsse=b)]
    t, s = lodo(pd.DataFrame(rows), default="A", pool=["A", "B"])
    assert t.loc["d3", "selected"] == "A"      # d3 held out: A wins on d1, d2
    assert t.loc["d1", "selected"] == "B"      # d1 held out: B wins on d2+d3 (mean 1.05 vs 1.25)


def test_diagnostics_columns_and_extended_registry():
    from drp_ap.diagnostics import diagnose
    from drp_ap.data import EXTENDED, ALL_NAMES
    X, y, K = S.multi_axial(n=800, d=15, r=2, ratio=0.9, seed=2)
    d = diagnose(X, K)
    for k in ("root_r", "root_ratio", "frac_not_u1", "init_ratio", "final_ratio", "gain_pct"):
        assert k in d and np.isfinite(d[k])
    assert len(EXTENDED) == 18 and len(ALL_NAMES) == 30
    from drp_ap.experiments import main_methods
    for k in ("SA-R best-of-10", "SA pool-3", "SA pool-10"):
        assert k in main_methods()


def test_fuzzy_partition_of_unity_and_hard_limit():
    from drp_ap.fuzzy import FuzzyDRPAPSA, fcm
    from drp_ap import DRPAPSA
    X, y, K = S.FAMILIES["subspace_gmm"](seed=0, r=3, decay=0.7)
    f = FuzzyDRPAPSA(m=2.0)
    C = f.fit_init(X, K)
    assert C.shape == (K, X.shape[1])
    assert np.allclose(f.U_.sum(1), 1.0) and (f.U_ >= -1e-12).all()
    # m -> 1 recovers hard SSE-arbitrated centres up to the weighted-median tie-breaking
    Ch = FuzzyDRPAPSA(m=1.0).fit_init(X, K)
    assert Ch.shape == C.shape
    C2, U2, J, it = fcm(X, C, m=2.0)
    assert np.isfinite(J) and np.allclose(U2.sum(1), 1.0)


def test_improvements_shapes_and_optimal_threshold():
    from drp_ap.improve import (DRPAPLook, DRPAPSAOpt, init_halving, optimal_threshold_split)
    rng = np.random.default_rng(0)
    p = rng.standard_normal(40)
    L, R = optimal_threshold_split(p)
    o = np.sort(p)
    sse = lambda a: ((a - a.mean()) ** 2).sum()
    best = min(sse(o[:j]) + sse(o[j:]) for j in range(1, 40))
    assert abs(sse(p[L]) + sse(p[R]) - best) < 1e-9
    X, y, K = S.FAMILIES["subspace_gmm"](seed=0, r=3, decay=0.7)
    for f in (DRPAPSAOpt(), DRPAPLook(), DRPAPLook(n_d2=2, random_state=0)):
        assert f.fit_init(X, K).shape == (K, X.shape[1])
    assert init_halving(X, K, np.random.default_rng(0)).shape == (K, X.shape[1])

In [ ]:
%%writefile tests/test_data.py
import numpy as np

from drp_ap import data as D


def test_manual_npz_and_stratified_subsample(tmp_path):
    rng = np.random.default_rng(0)
    X = rng.standard_normal((900, 7))
    y = np.repeat(np.array(["a", "b", "c"]), 300)
    np.savez(tmp_path / "Toy.npz", X=X, y=y)
    Xs, ys, K = D.load_dataset("Toy", data_dir=str(tmp_path), max_n=150)
    assert K == 3 and 140 <= len(Xs) <= 160
    assert set(np.bincount(ys)) <= set(range(45, 56))      # roughly stratified


def test_tfidf_lsa_pipeline():
    topics = [["goal match team score league"], ["stock market price trade bank"],
              ["gene protein cell biology dna"]]
    texts, labels = [], []
    rng = np.random.default_rng(0)
    for k, t in enumerate(topics):
        words = t[0].split()
        for _ in range(60):
            texts.append(" ".join(rng.choice(words, 8)) + " common filler words here")
            labels.append(k)
    Z = D._tfidf_lsa(texts, n_comp=5)
    assert Z.shape[0] == 180 and Z.shape[1] <= 5
    np.testing.assert_allclose(np.linalg.norm(Z, axis=1), 1.0, atol=1e-6)


def test_bundled_datasets_load():
    for name, d in [("Iris", 4), ("Wine", 13), ("BC", 30), ("Digits", 64)]:
        X, y, K = D.load_dataset(name)
        assert X.shape[1] == d and K >= 2


def test_pca_reduction_dimension():
    X = np.random.default_rng(0).standard_normal((120, 200))
    assert D._pca(X, 50).shape == (120, 50)

### Check that every file was written (re-run the cells above if anything is missing)

In [ ]:
import os
need = ['drp_ap/__init__.py', 'drp_ap/core.py', 'drp_ap/baselines.py', 'drp_ap/metrics.py', 'drp_ap/stats.py', 'drp_ap/synthetic.py', 'drp_ap/data.py', 'drp_ap/experiments.py', 'drp_ap/reporting.py', 'drp_ap/tuning.py', 'drp_ap/diagnostics.py', 'run_experiments.py', 'tools/merge_results.py', 'tests/test_drp_ap.py', 'tests/test_data.py']
missing = [f for f in need if not os.path.exists(f)]
assert not missing, "Missing files: %s  -> re-run the writefile cells above (Runtime > Run all)" % missing
print("OK: all", len(need), "files present")

## 2. Self-check (unit tests: Proposition 1 containment, Theorem 2 construction, shapes…)

In [ ]:
!python -m pytest -q tests

## 3. Quick demo — the deficiency region on one synthetic dataset

In [ ]:
import numpy as np, pandas as pd
from sklearn.cluster import KMeans
from drp_ap import DRPAP, DRPAPSA
from drp_ap import baselines as B, synthetic as S, metrics as M

RATIO = 0.95          # lambda2/lambda1 of the Theorem-2 construction (try 0.2, 0.5, 0.8, 0.95)
rows = []
for draw in range(10):                       # one draw is noisy: average over 10
    X, y, K = S.multi_axial(n=2000, d=20, r=2, ratio=RATIO, seed=draw)
    inits = {
      "k-means++":            lambda: B.init_kmeanspp(X, K, np.random.default_rng(draw)),
      "PCA-Part":             lambda: B.init_pca_part(X, K),
      "k-means++ MD":         lambda: B.init_kmeanspp_md(X, K, np.random.default_rng(draw)),
      "DRP-AP (paper)":       lambda: DRPAP(random_state=draw).fit_init(X, K),
      "DRP-AP-SA (T=0)":      lambda: DRPAPSA(local_iters=0).fit_init(X, K), # no split refinement
      "DRP-AP-SA (default)":  lambda: DRPAPSA().fit_init(X, K),              # T=5, top-3 axes (recommended)
    }
    for name, f in inits.items():
        C = f()
        km = KMeans(K, init=C, n_init=1).fit(X)
        rows.append((draw, name, M.sse(X, C), km.inertia_, km.n_iter_))
df = pd.DataFrame(rows, columns=["draw", "method", "init SSE", "final SSE", "iterations"])
df["final / best"] = df["final SSE"] / df.groupby("draw")["final SSE"].transform("min")
print(f"Mean over 10 draws, lambda2/lambda1 = {RATIO}")
print(df.groupby("method")[["init SSE", "final SSE", "final / best", "iterations"]].mean().round(3))

X, y, K = S.multi_axial(n=2000, d=20, r=2, ratio=RATIO, seed=0)
m = DRPAP(random_state=0); m.fit_init(X, K)
print("\nMode usage at each split (draw 0):", m.mode_counts_)
print(pd.DataFrame(m.trace_).round(3))

## 4. Configuration
* `QUICK=True`: 3 seeds, offline datasets (Iris, Wine, Breast Cancer, Digits) — smoke test.
* `QUICK=False`: 30 seeds on all 12 datasets (MNIST, Fashion-MNIST, COIL-20, Olivetti, 20 Newsgroups,
  Reuters R8, Isolet, UCI HAR are downloaded on first use; needs internet, available on Colab).
  `MAX_N` caps large datasets by stratified subsampling to keep runtime reasonable (set `0` for full size).
* If a dataset cannot be downloaded it is skipped with a warning; upload `data_cache/<Name>.npz`
  (arrays `X`, `y`) to supply it by hand.
* Optionally mount Google Drive and point `DATA_DIR` there to cache downloads between sessions.

In [ ]:
QUICK      = True        # False = full protocol
DATASETS   = "offline" if QUICK else "benchmark30"   # 30 datasets; or "all" (12 core) or a list
# To spread the 30 datasets over several Colab sessions use a list per session and a different OUT, e.g.
#   session 1: DATASETS = ["Iris","Wine","BC","Digits","MNIST","F-MNIST","COIL-20","Olivetti","20NG","R8","Isolet","HAR"]; OUT = "results_A"
#   session 2: DATASETS = ["glass","ecoli","yeast","vehicle","segment","satimage","pendigits","letter","waveform-5000"]; OUT = "results_B"
#   session 3: DATASETS = ["spambase","ionosphere","sonar","vowel","page-blocks","mfeat-karhunen","mfeat-fourier","optdigits","dermatology"]; OUT = "results_C"
# then merge them in section 7b (tools/merge_results.py).
OUT        = "results"
SEEDS      = 3 if QUICK else 30
MAX_N      = 20000       # 0 = no cap
EXTRAS     = True        # synthetic (x2) + ablations + sensitivity + stability + LODO tuning + diagnostics
RESTARTS   = True        # equal-compute baselines Greedy k-means++ best-of-3/10 (heavy: +13 Lloyd runs/seed)
DATA_DIR   = "data_cache"   # e.g. "/content/drive/MyDrive/drp_ap_data" after mounting Drive

# from google.colab import drive; drive.mount('/content/drive')

### Check which of the 30 datasets can be loaded (run this before the long run)
Downloads are cached in `DATA_DIR`. A dataset that fails (wrong name, network) is listed here;
drop `DATA_DIR/<name>.npz` (arrays `X`, `y`) to supply it by hand. The table is the paper's dataset table.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from drp_ap.data import ALL_NAMES, load_many, manifest
_names = ALL_NAMES if DATASETS in ("benchmark30", "extended") else (DATASETS if isinstance(DATASETS, list) else None)
if _names:
    _d = load_many(_names, DATA_DIR, MAX_N or None, verbose=False)
    print(f"loaded {len(_d)}/{len(_names)}; missing: {[n for n in _names if n not in _d]}")
    import pandas as pd; display(manifest(_d))

## 5. Run the experimental protocol

In [ ]:
import subprocess, shlex
cmd = f"python run_experiments.py --datasets {DATASETS if isinstance(DATASETS, str) else ' '.join(DATASETS)} " \
      f"--seeds {SEEDS} --max-n {MAX_N} --data-dir {DATA_DIR} --out {OUT} " \
      + ("--quick " if QUICK else "") + ("" if RESTARTS else "--no-restarts ") + ("--all-extras" if EXTRAS else "")
print(cmd)
!{cmd}

## 6. Results: tables, statistics and figures

In [ ]:
import pandas as pd, os
from IPython.display import Image, display
pd.set_option("display.width", 200)

print("Aggregate (mean over datasets and seeds; NSSE = normalised SSE, 1 = best observed)")
display(pd.read_csv("results/aggregate.csv", index_col=0).round(4))
print("Average ranks (Friedman/Nemenyi)")
display(pd.read_csv("results/average_ranks.csv", index_col=0).round(2)) if os.path.exists("results/average_ranks.csv") else None
print("Mode usage")
display(pd.read_csv("results/mode_usage.csv").round(3))
for f in ["cd_diagram.png", "synthetic_multiaxial.png"]:
    if os.path.exists("results/" + f):
        display(Image("results/" + f))
if os.path.exists("results/synthetic_verdicts.txt"):
    print(open("results/synthetic_verdicts.txt").read())
for f in ["ablation_table.csv", "stability.csv"]:
    if os.path.exists("results/" + f):
        print(f); display(pd.read_csv("results/" + f).round(4))

## 6b. Finish / resume the ablations and the sensitivity grid
If the main run stopped during the ablation stage (timeout, disconnect), do NOT re-run everything: the
main results are already saved. Run only the missing stages. Progress is saved after every dataset in
`ablation_raw.csv` / `sensitivity_raw.csv`, so the cell can be re-run until it prints `Done`.
The sensitivity grid (4 x 4 x 3 configurations x 5 seeds) is the slow one: restrict it to a few datasets.

In [ ]:
# ablations A1-A8 on all datasets (resumable):
!python run_experiments.py --datasets {DATASETS if isinstance(DATASETS, str) else ' '.join(DATASETS)} --max-n {MAX_N} --data-dir {DATA_DIR} --out {OUT} --skip-main --ablation
# sensitivity grid on a representative subset (resumable):
!python run_experiments.py --datasets Iris Wine BC Digits COIL-20 Olivetti glass vehicle segment --max-n {MAX_N} --data-dir {DATA_DIR} --out {OUT} --skip-main --sensitivity

## 7b. (optional) Merge several sessions into one benchmark
If you ran the datasets in groups (`results_A`, `results_B`, `results_C`), upload the three folders to
the same session and merge them: the statistics (Friedman, Nemenyi, Wilcoxon-Holm, LODO) are then computed over all datasets.

In [ ]:
# !python tools/merge_results.py --out results_all results_A results_B results_C

## 7. Export
`results/tab_sse.tex` and `results/tab_agg.tex` are ready-to-`\input` LaTeX tables that replace the
empty `Tables 2–3` of the paper (bold = best, † = significantly worse than DRP-AP, Wilcoxon–Holm).

In [ ]:
!zip -qr results.zip results
try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    pass